# In-context learning of a word graph: accuracy curve, belief-model fit and representations

*Author: Federico Viola · November 2026*

*Experiment 2 of [In-context graph learning in language models: induction circuits, learning curves and scaling laws](https://github.com/federicov99/in-context-graph-learning): the repository holds the write-up, the results and both notebooks.*

This notebook runs the main experiment of **Park et al. (2025), [*ICLR: In-Context Learning of
Representations*](https://arxiv.org/abs/2501.00070)** on a graph and model of your choice. It fits the resulting
accuracy curve with the belief-dynamics model of **Kowalyshyn et al. (2026), [*Belief or Circuitry? Causal Evidence
for In-Context Graph Learning*](https://arxiv.org/abs/2605.08405)** and compares that fit with Park et al.'s
two-line description of the same curve.

**The experiment.** We place unrelated nouns on the nodes of a graph (a square grid, a ring, a honeycomb, ...; or
Park et al.'s original 4×4 grid of words) and show the model a long random walk over the graph, written as words
separated by spaces: `apple bird car bird apple ...`. If the model learns the graph from the context, it should more
and more often predict one of the *neighbours* of the current word. The model's weights never change: whatever it
learns comes from the context.

**What you get**
1. **Accuracy vs context length** (section 7): the probability the model puts on valid next words, fitted with the
   belief model (parameters with bootstrap error bars) and with Park et al.'s two lines. The two descriptions are
   compared by held-out error and by BIC.
2. **A lookup reference** (section 8): how much of the graph the walk has already revealed at each point, computed
   from the walks alone. It lets you compare the model's timing with that of an idealised memoriser.
3. **A single-walk check** (section 9): does the S-shape of the averaged curve belong to the individual walks, or is
   it created by averaging walks that jump at different times? The check is calibrated with simulations built from
   the run's own noise.
4. **Representations** (sections 10–12): at several context lengths, each word's average activation in a late layer,
   projected onto 2 dimensions with PCA, and the **normalized Dirichlet energy**, a single number that is low when
   neighbouring words have similar activations.
5. **All the numbers behind the figures**, saved as CSV and NumPy files (section 13), so you can analyse them later
   without re-running the model.
6. **Optionally, a size scan** (section 14): the accuracy experiment for several graph sizes, with every fitted
   parameter plotted against the number of words.

**How to use it**
1. *Runtime → Change runtime type*: pick a GPU. The model table below shows which GPU each model needs.
2. **Llama and Gemma only:** add your Hugging Face token as a Colab secret named `HF_TOKEN` (key icon on the left,
   then switch on *Notebook access*). Your Hugging Face account must also have accepted that model's licence.
   Qwen and Mistral models need no token.
3. Edit the **Settings** cell, then choose *Runtime → Run all*.

The notebook does not touch your Google Drive. Results stay on the Colab machine and are offered as a download at the
end. Outside Colab, first install `torch`, `transformers>=4.56`, `accelerate`, `networkx`, `scipy`, `pandas`,
`matplotlib` and `tqdm`. Every random choice (words, walks, fits, simulations) follows from `SEED`, so a run can be
repeated, up to tiny numerical differences between GPUs.

**About the code.** The model runs through the standard Hugging Face implementation; nothing in its forward pass is
re-implemented. Software versions, seeds and the tests of the code are listed under *Technical details and
reproducibility* at the end. The notebook was generated with an AI model (Claude Opus 5.5) following the author's instructions, and checked by the author.

### Models (`MODEL_NAME`)

All are *base* (pretrained, not chat-tuned) models, like the one Park et al. used.

| `MODEL_NAME` | Parameters | Blocks | Max context | Token needed | Max words | Fits on |
|---|---|---|---|---|---|---|
| `meta-llama/Llama-3.2-1B` | 1.2 B | 16 | 128k | yes | 1,077 | T4, L4, A100 |
| `meta-llama/Llama-3.2-3B` | 3.2 B | 28 | 128k | yes | 1,077 | T4, L4, A100 |
| `meta-llama/Llama-3.1-8B` *(used by Park et al. and Kowalyshyn et al.)* | 8.0 B | 32 | 128k | yes | 1,077 | L4, A100 |
| `Qwen/Qwen2.5-1.5B` | 1.5 B | 28 | 32k | no | 1,077 | T4, L4, A100 |
| `Qwen/Qwen2.5-3B` | 3.1 B | 36 | 32k | no | 1,077 | T4, L4, A100 |
| `Qwen/Qwen2.5-7B` | 7.6 B | 28 | 128k | no | 1,077 | L4, A100 |
| `Qwen/Qwen2.5-14B` | 14.8 B | 48 | 128k | no | 1,077 | A100 |
| `Qwen/Qwen3-1.7B-Base` | 1.7 B | 28 | 32k | no | 1,077 | T4, L4, A100 |
| `Qwen/Qwen3-4B-Base` | 4.0 B | 36 | 32k | no | 1,077 | T4, L4, A100 |
| `Qwen/Qwen3-8B-Base` | 8.2 B | 36 | 32k | no | 1,077 | L4, A100 |
| `Qwen/Qwen3-14B-Base` | 14.8 B | 40 | 32k | no | 1,077 | A100 |
| `mistralai/Mistral-7B-v0.3` | 7.2 B | 32 | 32k | no | ≈ 480 | L4, A100 |
| `mistralai/Mistral-Nemo-Base-2407` | 12.2 B | 40 | 128k | no | ≈ 890 | A100 |
| `google/gemma-3-1b-pt` | 1.0 B | 26 | 32k | yes | ≈ 1,075 | T4 (in float32), L4, A100 |

**float16 on a T4.** If a model produces inf/NaN values in float16, the notebook stops and asks you to set
`DTYPE = "float32"`. That doubles the memory, so models above about 3B parameters no longer fit on a T4.

**Max words** is how many of the nouns are a single token for that model's tokenizer (≈ means within a few words;
the exact number is printed in section 4). It limits the graph size: for example, at most 21×21 = 441 words with
Mistral-7B, which has a small vocabulary.

**Memory.** The weights take about 2 bytes per parameter: 2.5 GB for a 1B model, 16 GB for 8B, 30 GB for 14B.
On top of that, the forward pass needs memory that grows with `NUM_TOKENS`: roughly 1–2 GB per 8,000 tokens for
models up to 8B. Usable memory is about 15 GB on a T4, 22 GB on an L4 and 40 or 80 GB on an A100.

**Speed (rough).** A 1B model takes about 1–2 s per 8,000-token walk on a T4. An 8B model takes about 2–4 s on an L4
and about 1 s on an A100. Longer walks cost more than proportionally, because attention grows with the square of the
length. The progress bar shows the real speed.

### Graphs (`GRAPH`)

| `GRAPH` | Shape | Neighbours per word | Words for `GRAPH_SIZE = n` | Used by |
|---|---|---|---|---|
| `"square"` | n×n square grid | 2–4 | n² | Park et al. (main experiment); Kowalyshyn et al. (4×4) |
| `"triangular"` | n×n rhombus of the triangular lattice: the square grid with one diagonal added to every square | 2–6 | n² | — |
| `"hexagonal"` | honeycomb of n×n hexagons, words on their corners | 2–3 | 2n(n+2): 16, 30, 48, 70, 96, ... for n = 2, 3, 4, 5, 6, ... | Park et al. (appendix; 30 words, n = 3) |
| `"ring"` | one closed loop | 2 | n² | Park et al.; Kowalyshyn et al. (16 words) |
| `"random"` | random graph, every word has 4 neighbours | 4 | n² | — (a graph with no spatial layout) |

The square, triangular and hexagonal graphs are built exactly as in the companion head-ablation notebook, so the
same `GRAPH_SIZE` gives the same graph there (its `ROWS` = `COLUMNS` = n).

The pictures are compared with a drawing of the true graph. For the lattices and the ring this is the natural
drawing. A random graph has no natural drawing, so it is compared with its *spectral* drawing: the graph Laplacian's
2nd and 3rd eigenvectors, which is the layout that minimizes the Dirichlet energy (Park et al., Sec. 5.1). If the
Laplacian's 3rd and 4th eigenvalues are nearly equal, that drawing is not well defined, and the notebook says so.

In [ ]:
# =============================== Settings ===============================
# Change these, then choose Runtime -> Run all.

MODEL_NAME = "meta-llama/Llama-3.2-1B"   # any model from the table above

GRAPH = "square"       # "square", "triangular", "hexagonal", "ring" or "random" (see the table above)
GRAPH_SIZE = 4         # n: n x n words for the square, triangular, ring and random graphs (4 -> 16 words);
                       # n x n hexagons with 2n(n + 2) words for the hexagonal one (2 -> 16, 3 -> 30 words)
WORDS = "random"       # "random" = random nouns from the word list; "park" = Park et al.'s 16 words in the layout
                       # of their Fig. 1 (apple bird car egg / house milk plane opera / ...); needs GRAPH = "square"
                       # and GRAPH_SIZE = 4
NUM_SEQUENCES = 32     # number of random walks. All walks use the same words, and results are averaged over them.
                       # The single-walk check (section 9) needs at least 16.
NUM_TOKENS = 4000      # length of each walk (one word = one token)

LAYER = "auto"         # which transformer block's output to look at (1 = first block).
                       # "auto" = the block at 80% of the depth (block 26 of 32 for Llama-3.1-8B,
                       # the layer Park et al. used).
WINDOW = "auto"        # how many recent tokens are averaged for each picture and energy point.
                       # "auto" = 3 tokens per word, at least 50 (50 for 16 words, as in Park et al.)
PCA_POINTS = "auto"    # context lengths for the activation pictures, e.g. [50, 200, 1000, 4000].
                       # "auto" = picked after the run, around the fitted transition point.
                       # You can also try other points later in the picture cell, without re-running the model.
SEED = 0               # change it to get a different set of words and different walks
DTYPE = "auto"         # "auto" = bfloat16 on L4/A100, float16 on T4. Use "float32" if you see NaN errors.
NUM_BOOTSTRAP = 100    # resamples used for the error bars of the fitted parameters

# Optional size scan (section 14): repeat the accuracy experiment for several GRAPH_SIZE values of the same GRAPH
# and plot every fitted parameter against the number of words.
SCAN_SIZES = []        # e.g. [3, 4, 5, 6, 8, 10]; empty = skip the scan
SCAN_TOKENS = None     # walk length for each size: None = NUM_TOKENS for all, or one number per size,
                       # e.g. [2000, 4000, 4000, 8000, 8000, 16000] (bigger graphs need longer walks)
SCAN_SEEDS = [0]       # word assignments per size, e.g. [0, 1, 2]. Each one draws different words (and walks),
                       # which shows how much the results depend on the choice of words.

## 1. Setup
This cell installs recent versions of the Hugging Face libraries (the newest models need them), reads your token
from the Colab secrets if there is one, and checks which GPU you have.

In [ ]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=4.56", "accelerate"],
                   check=True)
    from google.colab import userdata
    try:
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")    # read from the Secrets panel, never printed
    except Exception:
        print("No HF_TOKEN secret found. That's fine for Qwen and Mistral; Llama and Gemma need one.")

import json
import math
import shutil
import time

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import torch
import transformers
from matplotlib.collections import LineCollection
from scipy.optimize import minimize
from scipy.spatial import procrustes
from scipy.special import expit
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# Loading the model with from_pretrained(..., dtype=...) needs a recent version of transformers.
if tuple(int(part) for part in transformers.__version__.split(".")[:2]) < (4, 56):
    raise ImportError(f"This notebook needs transformers >= 4.56; this environment has {transformers.__version__}. "
                      "Install it with: pip install -U 'transformers>=4.56' accelerate")

if torch.cuda.is_available():
    DEVICE = "cuda"
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name} with {gpu.total_memory / 1e9:.0f} GB of memory")
else:
    DEVICE = "cpu"
    print("No GPU found: pick one under Runtime -> Change runtime type (on the CPU this is very slow).")

## 2. The word list
1,077 common English nouns, mostly everyday objects (`apple`, `bird`, `car`, ...), similar to the words Park et al.
used. No relations between them were built in, but a few are naturally related (for example `cat` and `dog`), and a
few are abstract (`solution`) or also names (`nova`). The words are put on the nodes at random, so such relations add
noise rather than structure; runs with several `SEED` values average over the choice of words.
Only the nouns that are a **single token** for the chosen model are used, so that one word = one token.

In [ ]:
NOUNS = """
world house system water head car hand body office room air art street road star picture key cup
blood dog oil door hair bed hospital paper earth internet sea ball box card piece wall river rock
eye addition island station ice computer photo sun hotel product brain floor skin fish machine
equipment camera cell coast foot hall coffee library queen airport engine solution structure lake
mouth surface arms hill agent prison stone window bag camp till cat dinner wood bridge garden
housing apple horse vehicle iron web arm bay mountain tea links valley boat museum steel beer moon
ocean chair planet restaurant sugar device platform pool fox hole leg studio apartment kitchen
aircraft truck bird bowl chicken meat neck stadium alcohol assembly humans wing facility bell
universe teeth bottle diet fruit lane tank transportation button rooms turkey carbon factory plate
rice roof bathroom weapon bike cake protein chest theatre cap bread eggs shoulder tower bush cabinet
chocolate highway auto guitar meal pants pocket chamber circuit bedroom colors egg hip
infrastructure railway bars castle palace temple bath muscle sand tape desk ear gym output pizza
android architecture belt flower kit lab stomach trash virus knife butter crystal layer pot sheet
bench cricket entrance makeup pen creek log sword tongue gene headquarters jean juice cotton theater
wolf bull glasses tail knee rings robin satellite bones innovation jacket tissue arsenal boards
footage laboratory mouse rocket cape hardware throat toy beef clay furniture jet mall globe stocks
copper vessel anime instrument brick garage lion oxygen powder shelter spider bullet dirt
robot shoe suspension laptop palm pole pub quarters rubber snake bin canal chips dish holder dose
flesh laser marijuana radar bacteria blade boot cargo corn nerve pie sheep tobacco basement ceiling
cherry cole costume garbage grain liver metro mixture oak soap vintage barrel couch lens ash casino
spirits bacon barrier cattle classroom colours costa ion medication portfolio salad soup twins
wallet workshop buffalo cow horn missile organ pig rod stairs zoo cabin cage closet disc eagle
embassy haven helicopter lip valve banner deer delta ham interface jewelry provisions chains drum
keyboard potato silk throne belly booth cliff derby lemon lung ore pine shark bee cafe dairy
receiver setup skull statue basin chick coke elephant fabric hub tooth basket beard cookie inn
rainbow shorts ambulance cigarette doll fiber grocery hull laundry brass lamp oven rabbit shooter
subway tablet tent airline collar nut sketch soda workplace cannabis chapel cloth container fur
kidney ladder lobby marc peninsula pill pork trunk cannon fridge goat gut helmet lamb machinery
mercury platinum shelf starter aluminum liquor salmon timber vitamin wheat wrist banana chin cottage
drone organs outlet portal puppy shades silicon vaccine vinyl asylum barn cathedral freight bean
cord hydrogen needle pond ankle armor berry poker steak artillery canvas canyon cheek fraction
marble underwear waist brake depot elevator locker mainland pencil plasma purse cement chess
generator idol junction vegetable cows ginger pistol sanctuary sleeve soundtrack spine airplane
debris garlic nova pasta runway sensor ammunition fountain wool brakes champagne monument particle
peanut petroleum slope vault wagon cocktail fist frog mosque moss nursery onion tomb umbrella bomber
candle coconut corridor mankind spoon terrace alley backyard billboard forehead marker marsh rubbish
salon sodium commodity curtain dough envelope granite greens lime maple pickup wheelchair builder
carriage gasoline ivory necklace porch tomato uranium elbow sights vanilla violet cone dessert
gateway livestock pumpkin reactor vodka calcium decoration electron enzyme hormone membrane nickel
pier cuisine dioxide insulin masterpiece pony ribbon shotgun sofa altar luggage puppet sausage
wardrobe receptor apparatus balcony drawer foam python supermarket sweater tractor trout whiskey
antenna dinosaur flats fortress herb jelly pathway peach stereo vein worm cane catalyst cedar crab
foil insect iris nitrogen shipment syrup turf volcano flagship glucose gown limestone spacecraft
tray baggage cereal ingredient jewellery serum skate supper torch violin aluminium arcade chili
glove hose lava magnet popcorn saddle thigh vine aisle atom cholesterol grease hallway jeep mayo
microphone petrol pixel strawberry sunglasses tile boiler calf cork detector recorder sticker
bracelet capsule cinnamon crust grape haircut limb pottery quartz beverage bumper chalk fallout
firearm mart mustard plaque shale shin squirrel stigma tinder jewel pulp scenery sidewalk staple
sushi tails timer bakery caffeine dell dolphin dove fibre fixture mattress molecule partition
replica yeast brewery bulb freezer hare oxide softball artery bamboo basil cellphone courtyard decor
driveway graffiti grenade helm lobster maze monastery nightclub panda seafood spaghetti woodland
camel charger cocoa comet compartment lever neon polymer rift sewer substrate tuna vinegar accessory
cache condo hash lizard mushroom podium turbine whisky womb yogurt bladder carrot furnace gluten
regulator vale cavity courthouse enclosure freeway jasmine lantern linen mosquito motel pavement
pigeon plateau pudding refrigerator resin simulator calculus cellar cockpit colon fetus mare mound
ness testosterone cortex leopard nucleus backdrop charcoal citrus clocks constituent courtroom dorm
duct emblem glacier kernel mosaic pineapple potassium scissors seaside shampoo staircase cartridge
cigar dungeon scooter venom abdomen aquarium consulate lithium mango porcelain vapor walnut bowel
brow cabbage chimney crater fragment heater meteor obstruction sedan sewage suitcase tavern titanium
vibes ware algae asbestos boutique burner chassis dagger fireplace headset kettle methane poultry
specs vase waterfall almond asphalt fungus lingerie marrow pastry residue salsa semiconductor
adrenaline canopy cruiser denim flute freshwater mast nicotine socket weeds weir cosmos doorway ling
parasite stripe broccoli cider destroyer doorstep hind mesa microscope moth raspberry axle barley
braces coco knob magnesium refinery siding slab webcam ammo blockbuster cameo caramel chromosome
convent crib formulation gravy placebo shutter spinach aperture char connector crate graft leash
pear pollen slit trivia wand antibody cleaners ethanol feeder flashlight instrumentation serpent
sesame tram amplifier avocado bathtub bunk gadget hamburger inlet lotion pinnacle roadside
synagogue cafeteria commuter cupboard neutron patio precursor rubble seam solvent squid tanker
blender chlorine hostel nutshell pendant repository trough accelerator acrylic ascent blanc
bookstore drawers facade farmhouse fertilizer filler flea footwear helium hemp medic nylon ozone
paving phosphate proton rotor standby stronghold turret bouquet clubhouse converter coop cucumber
foliage hoop ledge moderator projector revolver torso windshield abyss cassette ether latex
modem motif retina silicone thrift uterus broth chloride espresso pickle restroom satin scraps
snapshot biomass blouse compressor cougar electrode gland inhibitor leftovers workings dopamine
eyebrow launcher pigment relic roadway transformer ammonia dresser estrogen fleece payload
souvenir spaceship wastewater weaponry appliance casing collage cords graphite iceberg landfill
palate pedestal saliva shortcut treadmill cooker coupe embryo hardwood maize radiator bullpen
dashboard hairstyle mane mascara mitt montage steroid turquoise berg binder brig celery grinder
hinge photon potion takeaway labyrinth mailbox shoreline tendon viper adjunct ballpark bran
headphone peptide starch toxin
""".split()
print(len(NOUNS), "nouns")

## 3. Load the model
The weights are downloaded from Hugging Face (this takes a few minutes for the larger models) and loaded straight
onto the GPU. **bfloat16** and **float16** are 2-byte number formats: half the memory of the usual 4-byte float32,
and much faster on a GPU.

In [ ]:
def choose_dtype(setting):
    """Pick the number format for the weights."""
    if setting != "auto":
        return getattr(torch, setting)
    if DEVICE == "cpu":
        return torch.float32
    major_version, _ = torch.cuda.get_device_capability()
    # bfloat16 (the format these models were trained in) needs an Ampere-or-newer GPU such as the L4 or
    # A100 (version 8 or higher). The T4 (version 7.5) only has fast float16.
    return torch.bfloat16 if major_version >= 8 else torch.float16


dtype = choose_dtype(DTYPE)
if "gemma" in MODEL_NAME.lower() and dtype == torch.float16:
    print("Gemma's activations overflow float16, so it runs in float32 here (slower, twice the memory).")
    dtype = torch.float32

start_time = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dtype, device_map=DEVICE)
model.eval()                     # evaluation mode: turns off training-only behaviour such as dropout
torch.set_grad_enabled(False)    # we never train, so PyTorch doesn't need to keep track of gradients

# Some models post-process their outputs in ways this notebook does not reproduce.
for setting_name in ["final_logit_softcapping", "logits_scaling"]:
    if getattr(model.config, setting_name, None) not in (None, 1, 1.0):
        raise ValueError(f"{MODEL_NAME} uses {setting_name}, which this notebook doesn't support. "
                         "Please pick a model from the table.")

blocks = model.model.layers      # the transformer blocks, in order
num_layers = len(blocks)
layer = round(0.8 * num_layers) if LAYER == "auto" else int(LAYER)
if not 1 <= layer <= num_layers:
    raise ValueError(f"LAYER must be between 1 and {num_layers} for this model.")
hidden_size = model.config.hidden_size
vocab_size = model.lm_head.weight.shape[0]
max_context = getattr(model.config, "max_position_embeddings", None)

print(f"Loaded {MODEL_NAME} in {time.time() - start_time:.0f} s: {num_layers} blocks, "
      f"hidden size {hidden_size}, vocabulary {vocab_size:,} tokens, {str(dtype).replace('torch.', '')}.")
if DEVICE == "cuda":
    print(f"GPU memory used by the weights: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
print(f"We will look at the output of block {layer} of {num_layers}.")

## 4. The graph, its words and the random walks
We build the graph with the `networkx` library, number its nodes 0, 1, 2, ... and give each node a random noun.
A walk moves from a node to a random neighbour at every step; it may step straight back to the word it came from, as in Park et al.

As in Park et al., **all walks in a run use the same graph and the same words** and differ only in their path. This
lets us average each word's activations over all walks. To see a different set of words, change `SEED`. With
`WORDS = "park"` the grid is exactly the one in Park et al.'s Fig. 1 (`apple` top left, `phone` bottom right).
The walk is written with spaces between the words, as in all the runs here.

In [ ]:
def spectral_layout(adjacency):
    """A 2-D drawing of a graph from its Laplacian's 2nd and 3rd eigenvectors (used for graphs without geometry)."""
    A = adjacency.astype(float)
    laplacian = np.diag(A.sum(axis=1)) - A
    eigenvalues, eigenvectors = np.linalg.eigh(laplacian)     # eigenvalues in increasing order
    if len(eigenvalues) > 3 and eigenvalues[3] - eigenvalues[2] < 0.05 * eigenvalues[2]:
        print("Note: the Laplacian's 3rd and 4th eigenvalues are within 5% of each other, so the 2-D spectral "
              "drawing is not well defined; treat the PCA R² of this graph as indicative only.")
    return eigenvectors[:, 1:3]                              # skip the first one, which is constant


def build_graph(kind, size, seed):
    """Returns (adjacency, layout) for the graph of the given kind and size n: n x n nodes for the square, triangular,
    ring and random graphs, and n x n hexagons (2n(n + 2) nodes) for the hexagonal one.
    adjacency[i, j] is True when nodes i and j are neighbours.
    layout[i] is the (x, y) position of node i in a drawing of the graph."""
    target = size * size
    positions = None
    if kind == "square":
        graph = nx.grid_2d_graph(size, size)                          # nodes are (row, column) pairs
        positions = {node: (node[1], size - 1 - node[0]) for node in graph}     # x = column, row 0 at the top
    elif kind == "triangular":
        # An n x n rhombus of the triangular lattice: a square grid with the same diagonal added to every square,
        # drawn sheared so that every edge has length 1.
        graph = nx.grid_2d_graph(size, size)
        for r in range(size - 1):
            for c in range(1, size):
                graph.add_edge((r, c), (r + 1, c - 1))
        positions = {(r, c): (c + r / 2, (size - 1 - r) * math.sqrt(3) / 2) for r, c in graph}
    elif kind == "hexagonal":
        # n x n hexagons, with 2n(n + 2) nodes on their corners (n = 3: 30 nodes).
        graph = nx.hexagonal_lattice_graph(size, size, with_positions=True)
        positions = nx.get_node_attributes(graph, "pos")
    elif kind == "ring":
        graph = nx.cycle_graph(target)
        positions = {i: (math.cos(2 * math.pi * i / target), math.sin(2 * math.pi * i / target)) for i in graph}
    elif kind == "random":
        while True:                        # almost always connected at the first try; otherwise try another
            graph = nx.random_regular_graph(4, target, seed=seed)
            if nx.is_connected(graph):
                break
            seed += 1000
    else:
        raise ValueError(f'Unknown GRAPH "{kind}": use "square", "triangular", "hexagonal", "ring" or "random".')
    nodes = sorted(graph.nodes())                                    # fixes the node numbering 0, 1, 2, ...
    adjacency = nx.to_numpy_array(graph, nodelist=nodes) > 0
    layout = np.array([positions[node] for node in nodes]) if positions else spectral_layout(adjacency)
    return adjacency, layout


def token_ids_after_space(text):
    """Token ids of `text` when it comes after a space in running text, as the words do in our walks."""
    prefix = tokenizer.encode("the", add_special_tokens=False)
    with_text = tokenizer.encode("the " + text, add_special_tokens=False)
    return with_text[len(prefix):]


if GRAPH_SIZE < (2 if GRAPH == "hexagonal" else 3) or NUM_TOKENS < 50:
    raise ValueError("Use GRAPH_SIZE >= 3 (>= 2 for the hexagonal graph) and NUM_TOKENS >= 50.")
if WORDS not in ("random", "park"):
    raise ValueError('WORDS must be "random" or "park".')
if WORDS == "park" and (GRAPH != "square" or GRAPH_SIZE != 4):
    raise ValueError('WORDS = "park" is Park et al.\'s 4 x 4 grid: it needs GRAPH = "square" and GRAPH_SIZE = 4.')
rng = np.random.default_rng(SEED)              # all the randomness of the run comes from this generator
adjacency, layout = build_graph(GRAPH, GRAPH_SIZE, SEED)
num_words = len(adjacency)
degree = adjacency.sum(axis=1)

# Keep only the nouns that are exactly one token for this tokenizer.
single_token_words = {}
for word in NOUNS:
    ids = token_ids_after_space(word)
    if len(ids) == 1:
        single_token_words[word] = ids[0]
print(f"{len(single_token_words)} of the {len(NOUNS)} nouns are a single token for this model.")
if num_words > len(single_token_words):
    raise ValueError(f"This graph needs {num_words} words but only {len(single_token_words)} are available: "
                     "use a smaller GRAPH_SIZE.")

# Park et al.'s Fig. 1: their 4 x 4 grid, row by row (node r * 4 + c is at row r, column c).
PARK_GRID_WORDS = ["apple", "bird", "car", "egg",
                   "house", "milk", "plane", "opera",
                   "box", "sand", "sun", "mango",
                   "rock", "math", "code", "phone"]


def assign_words(adjacency, rng, choice="random"):
    """Put a single-token noun on every node: random nouns from the list (choice="random") or Park et al.'s
    4 x 4 grid (choice="park"). Returns
      words:            the word on each node
      word_tokens:      its token id
      neighbour_lists:  for each node, the array of its neighbours' node numbers
      neighbour_tokens: for each node, its neighbours' token ids, padded with -1 so all rows have the same length"""
    n = len(adjacency)
    if choice == "park":
        words = np.array(PARK_GRID_WORDS)
        token_ids = [token_ids_after_space(w) for w in words]
        not_single = [w for w, ids in zip(words, token_ids) if len(ids) != 1]
        if not_single:
            raise ValueError(f"For this model these Park et al. words are more than one token: {not_single}. "
                             'Use WORDS = "random" or another model.')
        word_tokens = np.array([ids[0] for ids in token_ids])
    else:
        words = rng.choice(list(single_token_words), size=n, replace=False)
        word_tokens = np.array([single_token_words[w] for w in words])
    neighbour_lists = [np.flatnonzero(adjacency[node]) for node in range(n)]
    neighbour_tokens = np.full((n, adjacency.sum(axis=1).max()), -1)
    for node in range(n):
        neighbour_tokens[node, :len(neighbour_lists[node])] = word_tokens[neighbour_lists[node]]
    return words, word_tokens, neighbour_lists, neighbour_tokens


def random_walk(length, neighbour_lists, rng):
    """Start at a random node, then step to a random neighbour, length - 1 times."""
    walk = [rng.integers(len(neighbour_lists))]
    for _ in range(length - 1):
        walk.append(rng.choice(neighbour_lists[walk[-1]]))
    return np.array(walk)


words, word_tokens, neighbour_lists, neighbour_tokens = assign_words(adjacency, rng, WORDS)


# Most models start every text with a special "beginning of sequence" token; Qwen models don't use one.
bos = [tokenizer.bos_token_id] if tokenizer.bos_token_id is not None else []

# Check that our token ids are exactly what the tokenizer produces for the walk written as text.
example = random_walk(12, neighbour_lists, rng)
example_text = " ".join(words[example])
assert token_ids_after_space(example_text) == word_tokens[example].tolist(), "tokenization mismatch"
if max_context is not None and NUM_TOKENS + len(bos) > max_context:
    raise ValueError(f"NUM_TOKENS = {NUM_TOKENS} is longer than this model's context ({max_context} tokens).")

# Colours used in all pictures: position around the ring, or height in the drawing for the other graphs.
colour_value = np.arange(num_words) if GRAPH == "ring" else layout[:, 1]
COLOUR_MAP = "twilight" if GRAPH == "ring" else "viridis"


def draw_graph(ax, xy, nodes, labels=False):
    """Draw the given nodes at positions xy (one row per node), joined by grey lines where the graph has an edge."""
    row_of = {node: k for k, node in enumerate(nodes)}
    edges = [[xy[row_of[a]], xy[row_of[b]]]
             for a, b in zip(*np.nonzero(np.triu(adjacency))) if a in row_of and b in row_of]
    ax.add_collection(LineCollection(edges, colors="0.75", linewidths=0.7, zorder=1))
    ax.scatter(xy[:, 0], xy[:, 1], c=colour_value[nodes], cmap=COLOUR_MAP, vmin=colour_value.min(),
               vmax=colour_value.max(), s=max(8, 2000 / num_words), zorder=2)
    if labels:
        for k, node in enumerate(nodes):
            ax.annotate(words[node], xy[k], fontsize=7, xytext=(3, 3), textcoords="offset points")
    ax.set_aspect("equal")
    ax.margins(0.12)                      # leave room for the labels
    ax.set_xticks([])
    ax.set_yticks([])


graph_figure, ax = plt.subplots(figsize=(5, 5))
draw_graph(ax, layout, np.arange(num_words), labels=num_words <= 40)
ax.set_title(f'{GRAPH} graph: {num_words} words, {int(adjacency.sum() // 2)} edges, '
             f'{degree.min()}-{degree.max()} neighbours per word', fontsize=9)
plt.show()
print("Example of a walk:", example_text, "...")

## 5. What we measure on each walk
**Accuracy.** After each word, the model gives a probability to every token in its vocabulary for the next one.
The *rule-following accuracy* is the total probability on the valid next words, i.e. the current word's neighbours
(Park et al., Sec. 4.1).

**Activations, without storing them all.** While the model runs, a *forward hook* copies the output of the chosen
block. We don't keep it. For each of about 30 context lengths `t` we add the activations of the last `WINDOW`
positions before `t` to a running sum for each word. All walks add into the same sums. At the end,
`sum / count` is each word's average activation just before `t`, averaged over all walks. That is what the
pictures and the energy are computed from, as in Park et al.

In [ ]:
# lm_head turns the final hidden states into one score (a "logit") per vocabulary token. Doing that for all
# positions at once would create a huge [positions x vocabulary] table, so we do CHUNK positions at a time
# (about 0.5 GB each).
CHUNK = max(64, 2 ** 27 // vocab_size)


def run_model_on_walk(walk, word_tokens, neighbour_tokens, keep_activations=True):
    """Feed one walk to the model. word_tokens and neighbour_tokens describe the graph (see assign_words).
    Returns
      accuracy:     accuracy[k] = probability of the valid next words after seeing k + 1 words, shape [len(walk) - 1]
      activations:  output of the chosen block at every word of the walk, shape [len(walk), hidden_size]
                    (None if keep_activations is False)"""
    input_ids = torch.tensor([bos + word_tokens[walk].tolist()], device=DEVICE)    # shape [1, positions]

    # A forward hook is a function PyTorch calls with a module's output whenever that module runs.
    captured = {}

    def save_block_output(module, inputs, output):
        hidden = output[0] if isinstance(output, tuple) else output     # some versions return a tuple
        captured["activations"] = hidden[0, len(bos):]                  # drop the batch dimension and the BOS

    hook = blocks[layer - 1].register_forward_hook(save_block_output) if keep_activations else None
    try:
        # model.model is the transformer without its last layer (lm_head); its output is the final hidden state.
        final_hidden = model.model(input_ids=input_ids, use_cache=False).last_hidden_state[0, len(bos):]
    finally:
        if hook is not None:
            hook.remove()
    if not torch.isfinite(final_hidden).all():
        raise FloatingPointError("The model produced inf/NaN values: set DTYPE = 'float32' and run again.")

    # The hidden state at word k predicts word k + 1, whose valid values are the neighbours of word k.
    valid_next = torch.as_tensor(neighbour_tokens[walk[:-1]], device=DEVICE)    # [len(walk) - 1, max degree]
    accuracy = []
    for start in range(0, len(walk) - 1, CHUNK):
        stop = min(start + CHUNK, len(walk) - 1)
        logits = model.lm_head(final_hidden[start:stop]).float()                # [chunk, vocabulary]
        # probability of a token = exp(logit) / sum(exp(all logits)); logsumexp computes the log of that sum
        log_total = torch.logsumexp(logits, dim=-1)
        ids = valid_next[start:stop]
        neighbour_logits = logits.gather(1, ids.clamp(min=0))                   # logits of the valid next words
        neighbour_logits[ids < 0] = -float("inf")                               # padding contributes nothing
        accuracy.append(torch.exp(torch.logsumexp(neighbour_logits, dim=-1) - log_total))
    return torch.cat(accuracy).cpu().numpy(), captured.get("activations")


# The context lengths at which we keep per-word sums: about 30 points evenly spaced on a log scale,
# plus any you asked for in PCA_POINTS.
window = max(50, 3 * num_words) if WINDOW == "auto" else int(WINDOW)
context_points = set(np.geomspace(8, NUM_TOKENS, 30).round().astype(int).tolist())
if PCA_POINTS != "auto":
    context_points |= {int(t) for t in PCA_POINTS if 2 <= t <= NUM_TOKENS}
context_points = sorted(context_points)

# window_sums[i, node] = sum of the activations of `node` in the window before context_points[i]
window_sums = torch.zeros(len(context_points), num_words, hidden_size, device=DEVICE)
window_counts = torch.zeros(len(context_points), num_words, device=DEVICE)
print(f"Window: {window} tokens. Per-word sums are kept at {len(context_points)} context lengths "
      f"({window_sums.numel() * 4 / 1e6:.0f} MB).")


def add_to_window_sums(walk, activations):
    """Add this walk's activations to the per-word sums of every context length."""
    walk_on_device = torch.as_tensor(walk, device=DEVICE)
    for i, t in enumerate(context_points):
        start = max(0, t - window)
        nodes = walk_on_device[start:t]
        # index_add_ adds row k of the activations to row nodes[k] of the sums, for every k
        window_sums[i].index_add_(0, nodes, activations[start:t].float())
        window_counts[i].index_add_(0, nodes, torch.ones(len(nodes), device=DEVICE))

## 6. Run the walks
This is the slow part. The progress bar shows the time per walk.

In [ ]:
accuracy = np.zeros((NUM_SEQUENCES, NUM_TOKENS - 1), dtype=np.float32)
all_walks = np.zeros((NUM_SEQUENCES, NUM_TOKENS), dtype=np.int16)

start_time = time.time()
for s in tqdm(range(NUM_SEQUENCES), desc="walks"):
    walk = random_walk(NUM_TOKENS, neighbour_lists, rng)
    accuracy[s], activations = run_model_on_walk(walk, word_tokens, neighbour_tokens)
    add_to_window_sums(walk, activations)
    all_walks[s] = walk
    del activations                          # free the GPU memory right away

run_seconds = time.time() - start_time
peak_memory_gb = torch.cuda.max_memory_allocated() / 1e9 if DEVICE == "cuda" else None
print(f"{NUM_SEQUENCES} walks of {NUM_TOKENS} tokens in {run_seconds:.0f} s ({run_seconds / NUM_SEQUENCES:.1f} s per walk).")
if peak_memory_gb:
    print(f"Peak GPU memory: {peak_memory_gb:.1f} GB")

## 7. Fit the accuracy curve
**Notation.** $N$ is the context length: the number of words of the walk seen so far (as in Kowalyshyn et al.).

**Belief model** (Kowalyshyn et al. 2026, Eq. 1; Eq. 11 in their appendix gives the single-graph form). The model
is treated as gathering evidence that the text follows a hidden graph:

$$\text{accuracy}(N) = p_0 + (q - p_0)\,\sigma\!\left(b + \gamma N^{1-\alpha}\right)$$

where $\sigma$ is the logistic sigmoid. The parameters are:
- $p_0$, $q$: accuracy before and after the model "commits" to the graph;
- $b$: prior log-odds, how sceptical it is at the start (negative = sceptical);
- $\gamma$: how fast evidence accumulates;
- $\alpha$: diminishing returns (0 = every new token is worth the same).

The **transition point** $N^* = (-b/\gamma)^{1/(1-\alpha)}$ is where the log-odds $b + \gamma N^{1-\alpha}$ cross
zero, i.e. where the curve is halfway between $p_0$ and $q$.

**How it is fitted.** Least squares with L-BFGS-B from 24 random starting points, because the fit has local minima,
within the search ranges of Kowalyshyn et al. (App. A.1): $b \in [-30, 30]$, $\gamma \in [10^{-6}, 50]$ (searched on
a log scale), $\alpha \in [0, 0.99]$, $q \in (p_0, 1]$. **One difference from the paper:** they estimate $p_0$ from
the first 100 tokens and fit only $b, \gamma, \alpha, q$. Here $p_0$ is fitted too, between 0 and 0.5, because on small
graphs the accuracy already rises within the first 100 tokens, which would inflate an estimate taken there.
The fitted points are the walk-averaged accuracy, averaged again over bins that are equally wide in $\log N$
(16 per decade, from $N = 5$), so that every decade of context counts equally. Error bars are 68% intervals from
refitting on walks resampled with replacement (bootstrap).

**Two lines** (Park et al., Sec. 4.1 and Fig. 5): a slow phase and a fast phase, two straight lines in $\log N$ that
meet where the slope increases. We call that breakpoint $t_c$ (the name is ours). When the accuracy levels off at the
end, a third, flatter line may cover the plateau, so that the levelling-off is not mistaken for the transition; BIC
decides whether it is used. The plateau line is our addition.

$t_c$ and $N^*$ measure different things: $t_c$ is where the rise *starts*, $N^*$ is its *midpoint*. They need not
scale in the same way with the size of the graph.

**Which description fits better?** The main criterion is held-out error: both are fitted on the average of a random
half of the walks and tested on the average of the other half (20 random splits). BIC is shown as well, but it assumes
the same noise level in every bin, which holds only roughly here.

**When to trust the fit.** $q$, $b$, $\gamma$ and $\alpha$ can only be pinned down once the curve has clearly levelled
off. As a rule of thumb the notebook calls a fit *reliable* when (i) by the end of the walk the accuracy has covered at
least 93% of the way from $p_0$ to $q$, (ii) no parameter other than $p_0$ sits at the edge of its search range, and
(iii) $N^*$ lies inside the range of context lengths that were measured. Otherwise it prints a warning and estimates,
from the current fit, how long the walks should be.

In [ ]:
def log_binned_mean(acc_matrix, bins_per_decade=16, smallest_context=5, positions=None):
    """Average the accuracy over walks, then over bins of context length that are equally wide on a log scale.
    positions: optional True/False mask selecting which positions to use (e.g. only the odd ones).
    Returns (bin centres, mean accuracy in each bin)."""
    mean_accuracy = acc_matrix.mean(axis=0)
    context = np.arange(1, len(mean_accuracy) + 1)       # accuracy[k] is measured after seeing k + 1 words
    if positions is None:
        positions = np.ones(len(context), dtype=bool)
    num_bins = max(8, int(bins_per_decade * np.log10(context[-1] / smallest_context)))
    edges = np.unique(np.round(np.geomspace(smallest_context, context[-1] + 1, num_bins + 1)))
    centres, means = [], []
    for low, high in zip(edges[:-1], edges[1:]):
        in_bin = (context >= low) & (context < high) & positions
        if in_bin.any():
            centres.append(np.exp(np.log(context[in_bin]).mean()))      # geometric centre of the bin
            means.append(mean_accuracy[in_bin].mean())
    return np.array(centres), np.array(means)


def belief_curve(N, p0, q, b, gamma, alpha):
    """Kowalyshyn et al. (2026), Eq. 1."""
    return p0 + (q - p0) * expit(b + gamma * N ** (1 - alpha))      # expit is the logistic sigmoid


def transition_point(b, gamma, alpha):
    """N*: the context length at which b + gamma * N^(1 - alpha) crosses zero (1 if it is positive from the start)."""
    return (-b / gamma) ** (1 / (1 - alpha)) if b < 0 else 1.0


# Search ranges of Kowalyshyn et al. (App. A.1), plus p0 between 0 and 0.5 (it is fitted here; see above).
# gamma is searched on a log scale.
NAMES = ["p0", "q", "b", "log10_gamma", "alpha"]
LOWER = [0.0, 0.0, -30.0, -6.0, 0.0]
UPPER = [0.5, 1.0, 30.0, np.log10(50), 0.99]


def fit_belief_curve(N, acc, rng, restarts=24, starts=()):
    """Least-squares fit of the belief curve. Tries `restarts` random starting points plus any parameter vectors given
    in `starts` (for example an earlier fit's "x"), and keeps the best result."""
    def squared_error(params):
        p0, q, b, log10_gamma, alpha = params
        if q <= p0:
            return 1e3                                    # the plateau must be above the starting level
        return np.sum((belief_curve(N, p0, q, b, 10 ** log10_gamma, alpha) - acc) ** 2)

    initial_points = [np.asarray(start, dtype=float) for start in starts]
    for _ in range(restarts):
        initial_points.append([rng.uniform(0, min(acc[0], 0.45)), rng.uniform(acc.max(), 1), rng.uniform(-15, 0),
                               rng.uniform(-4, 1), rng.uniform(0, 0.95)])
    best = None
    for start in initial_points:
        result = minimize(squared_error, start, method="L-BFGS-B", bounds=list(zip(LOWER, UPPER)))
        if best is None or result.fun < best.fun:
            best = result
    p0, q, b, log10_gamma, alpha = best.x
    at_bound = [name for name, value, low, high in zip(NAMES, best.x, LOWER, UPPER)
                if min(value - low, high - value) < 1e-3 * (high - low)]
    return {"p0": p0, "q": q, "b": b, "gamma": 10 ** log10_gamma, "alpha": alpha,
            "N*": transition_point(b, 10 ** log10_gamma, alpha), "sse": best.fun, "at_bound": at_bound, "x": best.x}


RELIABLE_PROGRESS = 0.93     # share of the rise from p0 to q that must be covered by the end of the walk


def fit_reliability(f, acc_matrix, measured_range, ignore=("p0",)):
    """Can a belief-curve fit of the walk-averaged acc_matrix be trusted (see the rule of thumb above)?
    measured_range: (smallest, largest) context length covered by the fitted points.
    ignore: parameters allowed to sit at the edge of their search range (p0 = 0 is common and harmless).
    Returns (reliable, progress, problems): progress is the share of the way from p0 to q covered by the end of the
    walk, problems a list of what went wrong."""
    final = acc_matrix[:, -max(20, acc_matrix.shape[1] // 50):].mean()
    progress = (final - f["p0"]) / (f["q"] - f["p0"])
    problems = [f"{name} at the edge of its search range" for name in f["at_bound"] if name not in ignore]
    if progress < RELIABLE_PROGRESS:
        problems.append(f"only {progress:.0%} of the rise to the plateau is covered by the end of the walk")
    if not measured_range[0] < f["N*"] < measured_range[1]:
        problems.append("N* lies outside the measured range of context lengths")
    return not problems, progress, problems


def tokens_needed(f, share=RELIABLE_PROGRESS):
    """Context length at which the fitted curve has covered `share` of its rise from p0 to q."""
    log_odds = math.log(share / (1 - share)) - f["b"]
    return (log_odds / f["gamma"]) ** (1 / (1 - f["alpha"])) if log_odds > 0 else 1.0


def segments_design(x, onset, plateau):
    """Columns of a continuous broken line in x = log10(N): level, slope of the slow phase (before onset), slope of
    the fast phase (onset to plateau) and, if plateau is a number rather than infinity, slope after the plateau."""
    columns = [np.ones_like(x), np.minimum(x - onset, 0), np.clip(x - onset, 0, plateau - onset)]
    if np.isfinite(plateau):
        columns.append(np.maximum(x - plateau, 0))
    return np.column_stack(columns)


def fit_two_lines(N, acc, num_candidates=60):
    """Park et al.'s slow and fast phase: two lines in log10(N) meeting at t_c, where the slope increases.
    A third line after a second breakpoint may take the plateau; BIC decides whether it is used.
    Breakpoints are found by trying num_candidates positions (and every later one for the plateau)."""
    x = np.log10(N)
    n = len(x)
    candidates = np.linspace(x[1], x[-2], num_candidates)
    best = {}                                        # best fit without ("two") and with ("three") a plateau line
    for i, onset in enumerate(candidates):
        for plateau in [np.inf] + list(candidates[i + 2:]):         # np.inf = no plateau line
            X = segments_design(x, onset, plateau)
            coefficients = np.linalg.lstsq(X, acc, rcond=None)[0]   # least-squares straight-line fit
            slow, fast = coefficients[1], coefficients[2]
            if fast <= slow:                                        # t_c must be a speed-up ...
                continue
            if np.isfinite(plateau) and coefficients[3] >= fast:    # ... and the plateau a slow-down
                continue
            sse = np.sum((X @ coefficients - acc) ** 2)
            kind = "two" if np.isinf(plateau) else "three"
            if kind not in best or sse < best[kind]["sse"]:
                best[kind] = {"t_c": 10 ** onset, "onset": onset, "plateau": plateau, "coefficients": coefficients,
                              "sse": sse, "num_params": 4 if kind == "two" else 6}
    if not best:                                     # the curve never speeds up: no transition to find
        return {"t_c": np.nan, "onset": 0.0, "plateau": np.inf, "coefficients": np.array([acc.mean(), 0.0, 0.0]),
                "sse": np.sum((acc - acc.mean()) ** 2), "num_params": 1}
    bic = {kind: n * np.log(f["sse"] / n) + f["num_params"] * np.log(n) for kind, f in best.items()}
    return best[min(bic, key=bic.get)]


def two_lines_curve(N, line_fit):
    """Values of a fitted broken line at context lengths N."""
    return segments_design(np.log10(N), line_fit["onset"], line_fit["plateau"]) @ line_fit["coefficients"]

In [ ]:
fit_rng = np.random.default_rng(SEED + 1)
context_bins, accuracy_bins = log_binned_mean(accuracy)
measured_range = (context_bins[0], NUM_TOKENS)        # context lengths covered by the fitted points
fit = fit_belief_curve(context_bins, accuracy_bins, fit_rng)
two_lines = fit_two_lines(context_bins, accuracy_bins)

# Bootstrap: refit on walks drawn at random with replacement, to see how much the parameters move. Each refit also
# starts from the full fit, so that an unlucky set of random starting points cannot throw it off.
bootstrap_fits = []
if NUM_SEQUENCES >= 2:
    for _ in tqdm(range(NUM_BOOTSTRAP), desc="bootstrap"):
        resampled = accuracy[fit_rng.integers(NUM_SEQUENCES, size=NUM_SEQUENCES)]
        bootstrap_fits.append(fit_belief_curve(*log_binned_mean(resampled), fit_rng, restarts=8, starts=[fit["x"]]))

meaning = {"p0": "accuracy before the transition",
           "q": "accuracy after the transition (plateau)",
           "b": "prior log-odds (more negative = more sceptical)",
           "gamma": "evidence rate",
           "alpha": "diminishing returns of evidence (0 = none)",
           "N*": "transition point: midpoint of the rise (tokens)"}
parameter_table = pd.DataFrame({
    "value": [fit[name] for name in meaning],
    "ci68_low": [np.percentile([f[name] for f in bootstrap_fits], 16) if bootstrap_fits else np.nan for name in meaning],
    "ci68_high": [np.percentile([f[name] for f in bootstrap_fits], 84) if bootstrap_fits else np.nan for name in meaning],
    "meaning": list(meaning.values())}, index=pd.Index(list(meaning), name="parameter"))
with pd.option_context("display.float_format", "{:.4g}".format):     # show 4 significant digits
    display(parameter_table)
print(f"Two-line fit: the rise starts at t_c = {two_lines['t_c']:.0f} tokens "
      f"({'with a plateau line' if two_lines['num_params'] == 6 else 'no plateau line'}).")

# Which description fits better? Main criterion: held-out error. Both are fitted on the average of a random half of
# the walks and tested on the average of the other half (20 random splits); this makes no assumption about the noise.
heldout = {"belief": [], "two_lines": []}
if NUM_SEQUENCES >= 4:
    for _ in range(20):
        order = fit_rng.permutation(NUM_SEQUENCES)
        train, test = order[: NUM_SEQUENCES // 2], order[NUM_SEQUENCES // 2:]
        x_train, y_train = log_binned_mean(accuracy[train])
        x_test, y_test = log_binned_mean(accuracy[test])
        f = fit_belief_curve(x_train, y_train, fit_rng, restarts=16)
        lines = fit_two_lines(x_train, y_train)
        heldout["belief"].append(np.mean((belief_curve(x_test, f["p0"], f["q"], f["b"], f["gamma"], f["alpha"]) - y_test) ** 2))
        heldout["two_lines"].append(np.mean((two_lines_curve(x_test, lines) - y_test) ** 2))
    belief_wins = np.mean(np.array(heldout["belief"]) < np.array(heldout["two_lines"]))
    print(f"Held-out error (fit on half the walks, tested on the other half): belief curve "
          f"{np.sqrt(np.mean(heldout['belief'])):.4f}, two lines {np.sqrt(np.mean(heldout['two_lines'])):.4f} "
          f"(RMS); the belief curve is better in {belief_wins:.0%} of the 20 splits.")
else:
    print("Held-out comparison skipped: it needs at least 4 walks.")

# Secondary check: BIC = n log(SSE / n) + (number of parameters) log n; lower is better. It assumes the same noise
# level in every bin, which holds only roughly here.
n = len(accuracy_bins)
bic_belief = n * np.log(fit["sse"] / n) + 5 * np.log(n)
bic_lines = n * np.log(two_lines["sse"] / n) + two_lines["num_params"] * np.log(n)
print(f"BIC(two lines) - BIC(belief curve) = {bic_lines - bic_belief:.0f}  "
      "(positive: the belief curve fits better; above 10 is usually read as a clear difference)")

# Can the parameters be trusted?
fit_reliable, progress, fit_problems = fit_reliability(fit, accuracy, measured_range)
print(f"By the end of the walk the accuracy has covered {progress:.0%} of the way from p0 = {fit['p0']:.2f} "
      f"to q = {fit['q']:.2f}.")
if fit_reliable:
    print("The fit passes the reliability checks.")
else:
    print("WARNING: the fit is probably not reliable: " + "; ".join(fit_problems) + ".")
    print(f"The current fit covers {RELIABLE_PROGRESS:.0%} of its rise at about {tokens_needed(fit):,.0f} tokens. That "
          "estimate comes from the same uncertain fit, so use walks of about twice that length.")

## 8. A lookup reference: how fast does the walk reveal the graph?
Before the model can know that `car` follows `bird`, the walk has to show it. For every position we compute the
**revealed share**: the fraction of the current word's neighbours that have already appeared *right after it* earlier
in the same walk. Because the next word is a uniformly random neighbour, this is also the probability that the next
step is a transition the context has already shown. It is computed from the walk alone, without the model.

The revealed share is a yardstick for *timing*, not a model of the network and not an upper bound on its accuracy:
a model can put all its probability on the few neighbours it has seen (and so score higher), or use transitions seen in
the opposite direction. So compare when the two curves rise and how that changes with graph size, not their heights.

The revealed share is fitted with the same belief curve and two lines as the accuracy. Its plateau is 1 by
construction, so $q$ at its upper limit is expected here. Its curve has no slow phase (it starts rising at once), so
its $N^*$ is more informative than its $t_c$. Because the walk often steps straight back, one neighbour of each word
is usually revealed within a few steps. On graphs with few neighbours per word, such as the ring, this early jump
already covers half of the neighbours, so the revealed share's $N^*$ mostly reflects that jump; the notebook then marks
it as not reliable, and the two curves are better compared directly. The ratio $N^*(\text{model}) / N^*(\text{revealed share})$ says how many
times more context the model needs to get halfway than the walk needs to show half of each word's neighbours.

In [ ]:
def revealed_share(walk, neighbour_lists):
    """Revealed share at every position k (after seeing k + 1 words, like `accuracy`): the fraction of the current
    word's neighbours that already appeared right after it earlier in the walk."""
    seen_after = [set() for _ in neighbour_lists]          # seen_after[v] = words seen right after v so far
    share = np.zeros(len(walk) - 1, dtype=np.float32)
    for k in range(len(walk) - 1):
        if k >= 1:
            seen_after[walk[k - 1]].add(walk[k])           # the step walk[k-1] -> walk[k] is now part of the context
        share[k] = len(seen_after[walk[k]]) / len(neighbour_lists[walk[k]])
    return share


EARLY_SHARE_LIMIT = 0.3      # above this average over the first bins, the revealed share's N* is not meaningful


def fit_lookup(share_matrix, rng):
    """Belief-curve and two-line fits of the walk-averaged revealed share, and whether its N* is meaningful.
    Returns (bin centres, bin means, belief fit, two-line fit, reliable, problems)."""
    x, y = log_binned_mean(share_matrix)
    belief, lines = fit_belief_curve(x, y, rng), fit_two_lines(x, y)
    # The plateau is 1 by construction, so q at its upper limit is expected here.
    reliable, _, problems = fit_reliability(belief, share_matrix, (x[0], share_matrix.shape[1] + 1), ignore=("p0", "q"))
    if y[:3].mean() > EARLY_SHARE_LIMIT:
        problems.append(f"the revealed share already averages {y[:3].mean():.2f} over the first bins, so its N* mostly "
                        "reflects the first back-and-forth steps")
        reliable = False
    return x, y, belief, lines, reliable, problems


lookup = np.array([revealed_share(walk, neighbour_lists) for walk in all_walks])
lookup_bins, lookup_bin_means, lookup_fit, lookup_lines, lookup_reliable, lookup_problems = fit_lookup(
    lookup, np.random.default_rng(SEED + 3))
final_share = lookup[:, -max(20, lookup.shape[1] // 50):].mean()

print(f"By the end of the walk, {final_share:.0%} of each word's neighbours have been shown right after it (average).")
print(f"Revealed share: midpoint N* = {lookup_fit['N*']:.0f} tokens (belief fit); onset t_c = {lookup_lines['t_c']:.0f} "
      "tokens (two lines; not very informative, as this curve has no slow phase).")
print(f"N*(model) / N*(revealed share) = {fit['N*'] / lookup_fit['N*']:.1f}"
      + ("" if fit_reliable and lookup_reliable else "   (not reliable: see the warnings)"))
if not lookup_reliable:
    print("The fit of the revealed share is not reliable: " + "; ".join(lookup_problems) + ".")

## 9. Single walks: is the S-shape real, or made by averaging?
An average of curves that each jump sharply, but at different times, also looks like a smooth S-curve. So a belief
curve that fits the *averaged* accuracy well says little about the individual walks. This section fits every walk on
its own and computes three diagnostics:

1. **Width ratio.** The *width* of a transition is the number of decades of context between 25% and 75% of the rise
   from $p_0$ to $q$. We divide the median single-walk width by the width of the averaged curve. Walks that jump sharply
   give a small ratio.
2. **Walk-to-walk spread of $N^*$** (in decades). Single-walk fits are noisy, so each walk is also fitted twice, once
   on its odd and once on its even positions, and the covariance of the two $\log_{10} N^*$ estimates is used: noise
   that differs between the two halves drops out. Only walks whose $N^*$ lies inside the measured range in both halves
   are used.
3. **Sharpening by alignment.** Each walk is shifted by its own $N^*$, measured on its odd positions, and the even
   positions are averaged again; we report the width after alignment divided by the width before. Averaged sharp jumps
   become sharp again once aligned; walks that each have the S-shape barely change.

**Calibration.** The values to expect depend on the noise, and noise shared by neighbouring positions does not drop
out of diagnostic 2. So the three diagnostics are also computed on two sets of simulated walks built from this run:
- *smooth walks*: every walk follows the averaged belief curve, plus the residual noise of one real walk around its
  own fit;
- *sharp jumps*: every walk jumps from $p_0$ to $q$ at a random time, with the jump times drawn so that on average
  the walks follow the averaged belief curve, plus the same residual noise.

For each diagnostic the notebook reports which simulation the real walks are closer to (or that the diagnostic cannot
tell the two apart), and gives a verdict by majority. The verdict is given only when there are at least 16 walks, the
averaged accuracy has levelled off inside the walk (section 7), and at least two diagnostics are informative. Per walk, the belief curve is also compared with the two lines,
by BIC and by held-out error (fit on the odd positions, test on the even ones, and the other way round).

In [ ]:
def transition_width(f):
    """Decades of context between 25% and 75% of the rise from p0 to q (the middle half of the transition)."""
    beta = 1 - f["alpha"]
    low, high = -math.log(3) - f["b"], math.log(3) - f["b"]     # where the log-odds reach -log 3 (25%) and log 3 (75%)
    if high <= 0:                                               # already past 75% of the rise at N = 1
        return np.nan
    start = (low / f["gamma"]) ** (1 / beta) if low > 0 else 1.0
    end = (high / f["gamma"]) ** (1 / beta)
    return math.log10(end) - math.log10(max(start, 1.0))


def mean_squared_error(f, x, y, model):
    prediction = (belief_curve(x, f["p0"], f["q"], f["b"], f["gamma"], f["alpha"]) if model == "belief"
                  else two_lines_curve(x, f))
    return np.mean((prediction - y) ** 2)


odd_positions = np.arange(NUM_TOKENS - 1) % 2 == 1
log_range = np.log10(measured_range)                     # a usable log10 N* must lie strictly inside this range


def single_walk_diagnostics(acc_matrix, average_fit, rng, label):
    """Fit every walk of acc_matrix on its own (all positions, odd positions, even positions) and compute the three
    diagnostics described above. average_fit is the belief-curve fit of the averaged curve.
    Returns (diagnostics, per-walk fits, data for the figure)."""
    walk_fits = []
    for s in tqdm(range(len(acc_matrix)), desc=label):
        one_walk = acc_matrix[s:s + 1]
        fits = {}
        for part, positions in [("all", None), ("odd", odd_positions), ("even", ~odd_positions)]:
            x, y = log_binned_mean(one_walk, bins_per_decade=10, positions=positions)
            fits[part] = {"x": x, "y": y, "fit": fit_belief_curve(x, y, rng, restarts=8)}
        walk_fits.append(fits)

    # 1. widths
    average_width = transition_width(average_fit)
    walk_widths = np.array([transition_width(w["all"]["fit"]) for w in walk_fits])
    median_walk_width = np.nanmedian(walk_widths) if np.isfinite(walk_widths).any() else np.nan
    width_ratio = median_walk_width / average_width

    # 2. walk-to-walk spread of log10 N*: covariance of the odd- and even-position estimates
    log_odd = np.log10([w["odd"]["fit"]["N*"] for w in walk_fits])
    log_even = np.log10([w["even"]["fit"]["N*"] for w in walk_fits])
    odd_inside = (log_odd > log_range[0]) & (log_odd < log_range[1])
    both_inside = odd_inside & (log_even > log_range[0]) & (log_even < log_range[1])
    spread = (math.sqrt(max(np.cov(log_odd[both_inside], log_even[both_inside])[0, 1], 0))
              if both_inside.sum() >= 4 else np.nan)

    # 3. plain and aligned averages of the even-position curves, on a common log grid. Each walk is shifted by the
    #    N* of its odd positions, so that no walk is aligned to the noise of the positions being averaged.
    plot = {"walk_widths": walk_widths, "walk_log_N*": np.log10([w["all"]["fit"]["N*"] for w in walk_fits]),
            "grid_x": None}
    alignment_ratio = np.nan
    usable = np.flatnonzero(odd_inside)
    if len(usable) >= 4:
        even_curves = [(walk_fits[s]["even"]["x"], walk_fits[s]["even"]["y"]) for s in usable]
        shift = log_odd[usable] - np.median(log_odd[usable])
        grid = np.linspace(np.log10(even_curves[0][0][0]), np.log10(even_curves[0][0][-1]), 120)
        aligned = np.array([np.interp(grid, np.log10(x) - d, y, left=np.nan, right=np.nan)
                            for (x, y), d in zip(even_curves, shift)])
        covered = np.mean(np.isfinite(aligned), axis=0) >= 0.9      # keep the range where 90% of the walks overlap
        if covered.sum() >= 10:
            grid_x = 10 ** grid[covered]
            plain = np.mean([np.interp(grid[covered], np.log10(x), y) for x, y in even_curves], axis=0)
            aligned_average = np.nanmean(aligned[:, covered], axis=0)
            plain_fit = fit_belief_curve(grid_x, plain, rng)
            aligned_fit = fit_belief_curve(grid_x, aligned_average, rng)
            alignment_ratio = transition_width(aligned_fit) / transition_width(plain_fit)
            plot.update(grid_x=grid_x, plain=plain, aligned=aligned_average, plain_fit=plain_fit,
                        aligned_fit=aligned_fit)

    diagnostics = {"width_ratio": width_ratio, "spread": spread, "alignment_ratio": alignment_ratio,
                   "average_width": average_width, "median_walk_width": median_walk_width,
                   "walks_used_for_spread": int(both_inside.sum())}
    return diagnostics, walk_fits, plot


# ---- the real walks
real, real_walk_fits, real_plot = single_walk_diagnostics(accuracy, fit, fit_rng, "single walks")

# Per walk: belief curve vs two lines, by BIC and by held-out error (odd positions -> even positions and back)
walk_rows = []
for s, fits in enumerate(real_walk_fits):
    lines = {part: fit_two_lines(fits[part]["x"], fits[part]["y"]) for part in fits}
    whole, odd, even = fits["all"]["fit"], fits["odd"]["fit"], fits["even"]["fit"]
    n = len(fits["all"]["y"])
    walk_rows.append({
        "walk": s, "N*": whole["N*"], "width_decades": transition_width(whole),
        "p0": whole["p0"], "q": whole["q"], "b": whole["b"], "gamma": whole["gamma"], "alpha": whole["alpha"],
        "N*_odd_positions": odd["N*"], "N*_even_positions": even["N*"],
        "bic_belief": n * np.log(whole["sse"] / n) + 5 * np.log(n),
        "bic_two_lines": n * np.log(lines["all"]["sse"] / n) + lines["all"]["num_params"] * np.log(n),
        "t_c": lines["all"]["t_c"],
        "heldout_mse_belief": (mean_squared_error(odd, fits["even"]["x"], fits["even"]["y"], "belief")
                               + mean_squared_error(even, fits["odd"]["x"], fits["odd"]["y"], "belief")) / 2,
        "heldout_mse_two_lines": (mean_squared_error(lines["odd"], fits["even"]["x"], fits["even"]["y"], "lines")
                                  + mean_squared_error(lines["even"], fits["odd"]["x"], fits["odd"]["y"], "lines")) / 2})
walk_table = pd.DataFrame(walk_rows)

# ---- calibration: two simulated sets of walks that keep the real noise
context = np.arange(1, NUM_TOKENS)                      # context length of each accuracy value
own_curves = np.array([belief_curve(context, *(w["all"]["fit"][k] for k in ("p0", "q", "b", "gamma", "alpha")))
                       for w in real_walk_fits])
residuals = accuracy - own_curves                       # each walk's noise around its own fit
average_curve = belief_curve(context, fit["p0"], fit["q"], fit["b"], fit["gamma"], fit["alpha"])
smooth_walks = np.clip(average_curve + residuals, 0, 1)

# Jump times T with P(T <= N) = sigmoid(b + gamma N^(1 - alpha)), so that the average of the jumps is the averaged
# belief curve: T solves b + gamma T^(1 - alpha) = logit(u) for a uniform random u.
simulation_rng = np.random.default_rng(SEED + 4)
u = simulation_rng.uniform(1e-9, 1 - 1e-9, size=NUM_SEQUENCES)
log_odds = np.log(u / (1 - u)) - fit["b"]
with np.errstate(over="ignore"):
    jump_times = np.where(log_odds > 0, (np.clip(log_odds, 0, None) / fit["gamma"]) ** (1 / (1 - fit["alpha"])), 1.0)
steps = (context[None, :] >= jump_times[:, None]).astype(float)
jump_walks = np.clip(fit["p0"] + (fit["q"] - fit["p0"]) * steps + residuals, 0, 1)

simulated = {}
for name, walks in [("smooth", smooth_walks), ("jumps", jump_walks)]:
    simulated_fit = fit_belief_curve(*log_binned_mean(walks), fit_rng, starts=[fit["x"]])
    simulated[name] = single_walk_diagnostics(walks, simulated_fit, fit_rng, f"simulated {name} walks")[0]
smooth, jumps = simulated["smooth"], simulated["jumps"]

labels = {"width_ratio": "1. median single-walk width / width of the averaged curve",
          "spread": "2. walk-to-walk spread of log10 N* (decades)",
          "alignment_ratio": "3. width after alignment / width before"}
calibration_table = pd.DataFrame(
    {"real walks": [real[k] for k in labels], "simulated smooth walks": [smooth[k] for k in labels],
     "simulated sharp jumps": [jumps[k] for k in labels]},
    index=pd.Index(list(labels.values()), name="diagnostic"))
votes = {}
for key in labels:
    measured, if_smooth, if_jumps = real[key], smooth[key], jumps[key]
    if not np.all(np.isfinite([measured, if_smooth, if_jumps])) or abs(if_smooth - if_jumps) < 0.1 * max(abs(if_smooth), abs(if_jumps)):
        votes[key] = "not informative"          # the two simulations give about the same value
    else:
        votes[key] = "smooth walks" if abs(measured - if_smooth) < abs(measured - if_jumps) else "sharp jumps"
calibration_table["real walks are closer to"] = [votes[k] for k in labels]
with pd.option_context("display.float_format", "{:.3f}".format):
    display(calibration_table)
print(f"Spread computed from {real['walks_used_for_spread']} of {NUM_SEQUENCES} walks (N* inside the measured range "
      "in both halves).")

informative = [v for v in votes.values() if v != "not informative"]
# The diagnostics use the widths and timing of the curves, not single parameters, so a parameter at the edge of its
# search range does not block the verdict; a curve that has not levelled off, or an N* outside the range, does.
curve_complete = progress >= RELIABLE_PROGRESS and measured_range[0] < fit["N*"] < measured_range[1]
if not curve_complete or NUM_SEQUENCES < 16 or len(informative) < 2:
    single_walk_verdict = "not decidable"
    reasons = (["the averaged accuracy has not levelled off inside the walk (section 7)"] * (not curve_complete)
               + ["it needs at least 16 walks"] * (NUM_SEQUENCES < 16)
               + ["fewer than two diagnostics separate the two simulations"] * (len(informative) < 2))
    print("Verdict: not decidable here: " + "; ".join(reasons) + ".")
elif informative.count("smooth walks") > informative.count("sharp jumps"):
    single_walk_verdict = "single walks have the S-shape"
    print(f"Verdict: on {informative.count('smooth walks')} of {len(informative)} informative diagnostics the real walks "
          "are closer to the smooth simulation: the single walks have the S-shape themselves; averaging did not "
          "create it.")
elif informative.count("sharp jumps") > informative.count("smooth walks"):
    single_walk_verdict = "S-shape made by averaging"
    print(f"Verdict: on {informative.count('sharp jumps')} of {len(informative)} informative diagnostics the real walks "
          "are closer to the sharp-jump simulation: the S-shape is at least partly made by averaging walks that jump "
          "at different times.")
else:
    single_walk_verdict = "mixed"
    print("Verdict: mixed: the informative diagnostics disagree.")
if single_walk_verdict != "not decidable" and not fit_reliable:
    print("Caution: the averaged fit has problems (" + "; ".join(fit_problems) + "), so the simulations are built on a "
          "curve that may not describe the data exactly.")

print(f"Per walk, belief curve vs two lines: better BIC in {np.mean(walk_table.bic_belief < walk_table.bic_two_lines):.0%} "
      f"of walks, better held-out error in "
      f"{np.mean(walk_table.heldout_mse_belief < walk_table.heldout_mse_two_lines):.0%} "
      f"(median held-out error ratio two lines / belief = "
      f"{np.median(walk_table.heldout_mse_two_lines / walk_table.heldout_mse_belief):.2f}).")

walk_figure, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))
widths = real_plot["walk_widths"]
ax1.hist(widths[np.isfinite(widths)], bins=20, color="0.6", label="real walks")
for value, colour, style, name in [(real["average_width"], "tab:orange", "-", "averaged curve"),
                                   (smooth["median_walk_width"], "tab:green", "--", "median, simulated smooth walks"),
                                   (jumps["median_walk_width"], "tab:purple", "--", "median, simulated sharp jumps")]:
    if np.isfinite(value):
        ax1.axvline(value, color=colour, ls=style, lw=2 if style == "-" else 1.5, label=name)
ax1.set_xlabel("single-walk transition width (decades, 25-75%)")
ax1.set_ylabel("walks")
ax1.legend(fontsize=7)
log_n_star = real_plot["walk_log_N*"]
ax2.hist(log_n_star[np.isfinite(log_n_star)], bins=20, color="0.6")
ax2.axvline(math.log10(fit["N*"]), color="tab:orange", lw=2, label="averaged curve")
ax2.set_xlabel("log10 of single-walk N*")
ax2.legend(fontsize=8)
if real_plot["grid_x"] is not None:
    for values, f, colour, name in [(real_plot["plain"], real_plot["plain_fit"], "tab:blue", "plain average"),
                                    (real_plot["aligned"], real_plot["aligned_fit"], "tab:red",
                                     "aligned by each walk's N*")]:
        ax3.plot(real_plot["grid_x"], values, ".", ms=3, color=colour)
        ax3.plot(real_plot["grid_x"], belief_curve(real_plot["grid_x"], f["p0"], f["q"], f["b"], f["gamma"], f["alpha"]),
                 color=colour, label=f"{name}: width {transition_width(f):.2f} decades")
    ax3.legend(fontsize=8)
else:
    ax3.set_title("too few walks with a usable N* to align", fontsize=9)
ax3.set_xscale("log")
ax3.set_ylim(0, 1)
ax3.set_xlabel("context length (tokens)")
ax3.set_ylabel("accuracy (even positions)")
walk_figure.suptitle("Single walks vs the averaged curve", fontsize=10)
walk_figure.tight_layout()
plt.show()

## 10. Representations at every stored context length
For each stored context length, we take every word's average activation (over the window and all walks) and compute:

**Normalized Dirichlet energy** (Kowalyshyn et al., Eq. 6):

$$E_\text{norm} = \frac{\tfrac12\sum_{ij} A_{ij}\,\lVert m_i - m_j\rVert^2}{\sum_i d_i\,\lVert m_i - \bar m\rVert^2}$$

Here $m_i$ is word $i$'s mean activation, $A$ the graph's adjacency matrix, $d_i$ the number of neighbours of word $i$,
and $\bar m$ the mean weighted by $d_i$ (the paper calls it the degree-weighted mean). The numerator is small when
neighbouring words have similar activations. For randomly placed points the ratio is about 1.

**PCA picture and its R²**: the words' coordinates on the two main directions of variation. They are rotated, mirrored
and scaled to line up with the drawing of the true graph as well as possible (Procrustes). R² = 1 means a perfect
match.

Both get a **baseline**: the same quantity with the words placed on the graph at random (average of 20 shuffles).
The gap between measurement and baseline is the part that reflects the graph. With few words even random points
reach a fairly high R², which the baseline makes visible.

**Two caveats, mainly for large graphs.**
- *Graph distance and time go together.* On a large graph a random walk stays in one region for a long time, so words
  that are close on the graph also tend to occur at similar times. Any activation that carries information about
  *when* a word occurred (its position, or how recently it was seen) then also looks smooth on the graph, and the
  random-placement baseline does not remove this. On small graphs the walk crosses the whole graph within a few
  dozen steps and the effect is small; on large graphs, read low energy and graph-like pictures with this in mind.
- *Long windows.* With `WINDOW = "auto"` the window is 3 tokens per word (at least 50), so that each word appears a few
  times in it. On large graphs the window is long, and each point averages over a wide range of context lengths.

In [ ]:
def word_means(i):
    """Average activation of each word in the window before context_points[i], pooled over all walks.
    Returns the means of the words that occurred in that window, and their node numbers."""
    seen = torch.nonzero(window_counts[i] > 0).flatten()
    means = window_sums[i, seen] / window_counts[i, seen, None]
    return means, seen.cpu().numpy()


def normalized_energy(squared_distances, means, adjacency_matrix):
    """Normalized Dirichlet energy (formula above). squared_distances[i, j] = |m_i - m_j|^2."""
    A = torch.as_tensor(adjacency_matrix, dtype=torch.float32, device=means.device)
    node_degree = A.sum(dim=1)
    neighbour_term = 0.5 * (A * squared_distances).sum()
    weighted_mean = (node_degree[:, None] * means).sum(dim=0) / node_degree.sum()
    spread = (node_degree * ((means - weighted_mean) ** 2).sum(dim=1)).sum()
    return (neighbour_term / spread).item()


def pca_2d(means):
    """Coordinates of each word along the two directions in which the means vary most."""
    centred = means - means.mean(dim=0)
    U, S, _ = torch.linalg.svd(centred, full_matrices=False)
    return (U[:, :2] * S[:2]).cpu().numpy()


num_points = len(context_points)
words_seen = np.zeros(num_points, dtype=int)
energy = np.full(num_points, np.nan)
energy_baseline = np.full(num_points, np.nan)
energy_baseline_std = np.full(num_points, np.nan)
pca_r2 = np.full(num_points, np.nan)
pca_r2_baseline = np.full(num_points, np.nan)
pca_coordinates = np.full((num_points, num_words, 2), np.nan)    # PC1, PC2 of each word (NaN = not seen yet)
pca_aligned = np.full((num_points, num_words, 2), np.nan)        # the same, lined up with the true drawing
baseline_rng = np.random.default_rng(SEED + 2)

for i in range(num_points):
    means, seen = word_means(i)
    words_seen[i] = len(seen)
    sub_adjacency = adjacency[np.ix_(seen, seen)]           # the graph restricted to the words seen so far
    if len(seen) < 4 or not sub_adjacency.any():
        continue
    centred = means - means.mean(dim=0)

    squared_distances = torch.cdist(centred, centred) ** 2
    energy[i] = normalized_energy(squared_distances, centred, sub_adjacency)
    shuffles = [baseline_rng.permutation(len(seen)) for _ in range(20)]     # random placements of the words
    shuffled_energies = [normalized_energy(squared_distances, centred, sub_adjacency[np.ix_(p, p)]) for p in shuffles]
    energy_baseline[i], energy_baseline_std[i] = np.mean(shuffled_energies), np.std(shuffled_energies)

    coordinates = pca_2d(centred)
    true_xy = layout[seen]
    try:
        # procrustes returns both point sets centred and scaled, the second one rotated onto the first
        _, aligned, disparity = procrustes(true_xy, coordinates)
    except ValueError:                                       # degenerate case, e.g. all points identical
        continue
    pca_r2[i] = 1 - disparity
    pca_r2_baseline[i] = np.mean([1 - procrustes(true_xy[p], coordinates)[2] for p in shuffles])
    pca_coordinates[i, seen] = coordinates
    pca_aligned[i, seen] = aligned

representation_table = pd.DataFrame({
    "context": context_points, "words_seen": words_seen,
    "energy": energy, "energy_baseline": energy_baseline, "energy_baseline_std": energy_baseline_std,
    "pca_r2": pca_r2, "pca_r2_baseline": pca_r2_baseline})
display(representation_table.round(3))

## 11. Summary figure
Top: accuracy with both fits, and the revealed share of section 8 in grey. Middle: normalized Dirichlet energy.
Bottom: how graph-like the PCA picture is. The dash-dotted line marks the fitted transition point $N^*$.

In [ ]:
summary_figure, (ax_acc, ax_energy, ax_r2) = plt.subplots(3, 1, figsize=(9, 10), sharex=True)
N_smooth = np.geomspace(context_bins[0], context_bins[-1], 400)
chance = (degree[all_walks[:, :-1]] / num_words).mean()     # guessing a random word of the graph

ax_acc.plot(context_bins, accuracy_bins, "o", ms=4, color="tab:blue", label="measured (mean over walks)")
ax_acc.plot(N_smooth, belief_curve(N_smooth, fit["p0"], fit["q"], fit["b"], fit["gamma"], fit["alpha"]),
            color="tab:orange", lw=2, label="belief curve fit")
ax_acc.plot(N_smooth, two_lines_curve(N_smooth, two_lines), "--", color="tab:red", lw=1, label="two-line fit")
if np.isfinite(two_lines["t_c"]):
    ax_acc.axvline(two_lines["t_c"], color="tab:red", lw=1, ls=":", label="t_c (two-line fit: the rise starts)")
ax_acc.plot(lookup_bins, lookup_bin_means, "-", color="0.5", lw=1.2,
            label=f"revealed share (lookup reference, section 8), N* = {lookup_fit['N*']:.0f}")
ax_acc.axhline(chance, color="0.6", ls=":", lw=1, label="chance (random word of the graph)")
ax_acc.set_ylim(0, 1.02)
ax_acc.set_ylabel("rule-following accuracy")
ax_acc.set_title(f"{MODEL_NAME}, {GRAPH} graph with {num_words} words, {NUM_SEQUENCES} walks: N* = {fit['N*']:.0f}, "
                 f"b = {fit['b']:.2f}, γ = {fit['gamma']:.3g}, α = {fit['alpha']:.2f}, q = {fit['q']:.2f}"
                 + ("" if fit_reliable else "  [fit not reliable: see section 7]"), fontsize=9)

ax_energy.plot(context_points, energy, "o-", ms=3, color="tab:green", label=f"block {layer} activations")
ax_energy.fill_between(context_points, energy_baseline - energy_baseline_std, energy_baseline + energy_baseline_std,
                       color="0.85")
ax_energy.plot(context_points, energy_baseline, "--", color="0.5", label="baseline: words placed at random")
ax_energy.set_ylabel("normalized Dirichlet energy")

ax_r2.plot(context_points, pca_r2, "o-", ms=3, color="tab:purple", label="PCA picture vs true drawing")
ax_r2.plot(context_points, pca_r2_baseline, "--", color="0.5", label="baseline: words placed at random")
ax_r2.set_ylim(0, 1)
ax_r2.set_ylabel("PCA R²")
ax_r2.set_xlabel("context length (tokens)")
ax_r2.set_xscale("log")
for ax in (ax_acc, ax_energy, ax_r2):
    if 1 < fit["N*"] < NUM_TOKENS:
        ax.axvline(fit["N*"], color="tab:orange", lw=1, ls="-.", label="N* (belief fit)")
    ax.legend(fontsize=8)
summary_figure.tight_layout()
plt.show()

## 12. Pictures of the representations
Each panel shows the words' average activations (block `LAYER`, window `WINDOW`, all walks) at one context length,
projected with PCA and lined up with the true drawing of the graph (section 4). Colours are the same as in that
drawing, and grey lines join words that are neighbours in the graph.

By default the points are chosen around the fitted transition: $N^*/4$, $N^*$, $4N^*$ and the end of the walk. To try
other points, edit `points_to_show` below and re-run only this cell. The nearest stored context length is used.

In [ ]:
points_to_show = PCA_POINTS        # e.g. [50, 200, 1000, 4000]; "auto" = around the transition point


def suggested_points():
    """Before, at and after the fitted transition point, and the end of the walk."""
    n_star = fit["N*"]
    if not (np.isfinite(n_star) and context_points[0] < n_star < NUM_TOKENS):
        n_star = math.sqrt(context_points[0] * NUM_TOKENS)       # no usable N*: the middle of the range (log scale)
    return [n_star / 4, n_star, 4 * n_star, NUM_TOKENS]


def nearest_stored_index(t):
    """Index of the stored context length closest to t (comparing on a log scale)."""
    return int(np.argmin(np.abs(np.log(context_points) - np.log(t))))


requested = suggested_points() if points_to_show == "auto" else points_to_show
shown_indices = sorted({nearest_stored_index(t) for t in requested})
print("Showing context lengths:", [context_points[i] for i in shown_indices])

pca_figure, axes = plt.subplots(1, len(shown_indices), figsize=(4.2 * len(shown_indices), 4.6), squeeze=False)
for ax, i in zip(axes[0], shown_indices):
    seen = np.flatnonzero(~np.isnan(pca_aligned[i, :, 0]))
    if len(seen) == 0:
        ax.set_title(f"t = {context_points[i]}: too few words seen", fontsize=9)
        ax.axis("off")
        continue
    draw_graph(ax, pca_aligned[i, seen], seen, labels=num_words <= 25)
    ax.set_title(f"t = {context_points[i]}: {len(seen)} words\nR² = {pca_r2[i]:.2f} "
                 f"(random placement: {pca_r2_baseline[i]:.2f})", fontsize=9)
pca_figure.suptitle(f"{MODEL_NAME}, block {layer}: PCA of word means (window {window} tokens, {NUM_SEQUENCES} walks)",
                    fontsize=10)
pca_figure.tight_layout()
plt.show()

## 13. Save the figures and the data behind them
Everything goes into one folder, which is also zipped and, in Colab, downloaded to your computer. `README.txt`
in the folder describes each file. In short:

| File | Contents |
|---|---|
| `accuracy_mean.csv` | mean accuracy (and its standard error) and mean revealed share at every context length, with both fitted curves |
| `accuracy_binned.csv` | the log-binned points that were fitted and plotted |
| `fit_parameters.csv`, `bootstrap_fits.csv` | the belief-curve parameters with 68% intervals, and every bootstrap refit |
| `per_walk_fits.csv` | the fit of every single walk (section 9) |
| `single_walk_calibration.csv` | the three single-walk diagnostics for the real walks and the two simulations |
| `representations.csv` | energy, PCA R² and their baselines at every stored context length |
| `pca_coordinates.csv` | PCA coordinates of every word at every stored context length (raw and lined up) |
| `graph_nodes.csv`, `graph_edges.csv` | the graph: words, token ids, drawing positions, edges |
| `data.npz` | NumPy arrays: accuracy and revealed share of every walk at every position, the walks, the graph, the PCA coordinates and (if not too large) every word's mean activation at every stored context length |
| `summary.json` | settings, model, fitted parameters, model comparison, lookup reference, single-walk verdict, run time |
| `*.png` | the figures |

The word means let you compute any other representation measure later, without the model. They are saved in
float16 when they take less than `MAX_WORD_MEANS_GB`.

In [ ]:
MAX_WORD_MEANS_GB = 1.0

run_name = (f"{MODEL_NAME.split('/')[-1]}_{GRAPH}{num_words}{'_park_words' if WORDS == 'park' else ''}"
            f"_tokens{NUM_TOKENS}_walks{NUM_SEQUENCES}_seed{SEED}")
out_dir = os.path.join("results", run_name)
os.makedirs(out_dir, exist_ok=True)
context = np.arange(1, NUM_TOKENS)                  # context length of each accuracy value

# ---- accuracy and lookup reference
pd.DataFrame({
    "context": context,
    "mean_accuracy": accuracy.mean(axis=0),
    "standard_error": accuracy.std(axis=0) / math.sqrt(NUM_SEQUENCES),
    "belief_fit": belief_curve(context, fit["p0"], fit["q"], fit["b"], fit["gamma"], fit["alpha"]),
    "two_line_fit": two_lines_curve(context, two_lines),
    "mean_revealed_share": lookup.mean(axis=0),
}).to_csv(os.path.join(out_dir, "accuracy_mean.csv"), index=False)
pd.DataFrame({
    "context_bin_centre": context_bins,
    "mean_accuracy": accuracy_bins,
    "belief_fit": belief_curve(context_bins, fit["p0"], fit["q"], fit["b"], fit["gamma"], fit["alpha"]),
    "two_line_fit": two_lines_curve(context_bins, two_lines),
    "mean_revealed_share": lookup_bin_means,         # same bins as the accuracy
}).to_csv(os.path.join(out_dir, "accuracy_binned.csv"), index=False)
parameter_table.to_csv(os.path.join(out_dir, "fit_parameters.csv"))
pd.DataFrame([{name: f[name] for name in meaning} for f in bootstrap_fits]).to_csv(
    os.path.join(out_dir, "bootstrap_fits.csv"), index=False)
walk_table.to_csv(os.path.join(out_dir, "per_walk_fits.csv"), index=False)
calibration_table.to_csv(os.path.join(out_dir, "single_walk_calibration.csv"))

# ---- representations
representation_table.to_csv(os.path.join(out_dir, "representations.csv"), index=False)
rows = []
for i, t in enumerate(context_points):
    for node in np.flatnonzero(~np.isnan(pca_coordinates[i, :, 0])):
        rows.append({"context": t, "node": node, "word": words[node],
                     "pc1": pca_coordinates[i, node, 0], "pc2": pca_coordinates[i, node, 1],
                     "aligned_x": pca_aligned[i, node, 0], "aligned_y": pca_aligned[i, node, 1]})
pd.DataFrame(rows).to_csv(os.path.join(out_dir, "pca_coordinates.csv"), index=False)

# ---- graph
pd.DataFrame({"node": np.arange(num_words), "word": words, "token_id": word_tokens, "degree": degree,
              "layout_x": layout[:, 0], "layout_y": layout[:, 1]}).to_csv(
    os.path.join(out_dir, "graph_nodes.csv"), index=False)
edge_a, edge_b = np.nonzero(np.triu(adjacency))
pd.DataFrame({"node_a": edge_a, "node_b": edge_b}).to_csv(os.path.join(out_dir, "graph_edges.csv"), index=False)

# ---- raw arrays
arrays = {"accuracy": accuracy, "revealed_share": lookup, "walks": all_walks, "context_points": np.array(context_points),
          "adjacency": adjacency, "layout": layout, "words": words, "word_tokens": word_tokens,
          "pca_coordinates": pca_coordinates, "pca_aligned": pca_aligned, "word_counts": window_counts.cpu().numpy()}
word_means_gb = window_sums.numel() * 2 / 1e9
if word_means_gb <= MAX_WORD_MEANS_GB:
    counts = window_counts.clamp(min=1)[:, :, None]
    arrays["word_means"] = (window_sums / counts).half().cpu().numpy()   # zero where the word wasn't seen
else:
    print(f"Word means not saved: {word_means_gb:.1f} GB > MAX_WORD_MEANS_GB = {MAX_WORD_MEANS_GB}.")
np.savez_compressed(os.path.join(out_dir, "data.npz"), **arrays)

# ---- settings and results
summary = {
    "settings": {"model": MODEL_NAME, "graph": GRAPH, "graph_size": GRAPH_SIZE, "num_words": num_words, "words": WORDS,
                 "num_sequences": NUM_SEQUENCES, "num_tokens": NUM_TOKENS, "layer": layer, "num_layers": num_layers,
                 "window": window, "seed": SEED, "dtype": str(dtype), "bos_token": bool(bos)},
    "belief_fit": {name: float(fit[name]) for name in meaning},
    "belief_fit_reliable": bool(fit_reliable),
    "belief_fit_problems": fit_problems,
    "progress_towards_plateau": float(progress),
    "two_line_onset_t_c": float(two_lines["t_c"]),
    "two_line_fit_has_plateau_line": bool(two_lines["num_params"] == 6),
    "heldout_rmse_belief": float(np.sqrt(np.mean(heldout["belief"]))) if heldout["belief"] else None,
    "heldout_rmse_two_lines": float(np.sqrt(np.mean(heldout["two_lines"]))) if heldout["two_lines"] else None,
    "bic_two_lines_minus_belief": float(bic_lines - bic_belief),
    "lookup_reference": {"revealed_share_N*": float(lookup_fit["N*"]), "revealed_share_t_c": float(lookup_lines["t_c"]),
                         "fit_reliable": bool(lookup_reliable), "final_revealed_share": float(final_share),
                         "model_N*_over_revealed_share_N*": float(fit["N*"] / lookup_fit["N*"])},
    "single_walks": {"verdict": single_walk_verdict,
                     "walks_used_for_spread": real["walks_used_for_spread"],
                     "diagnostics": {key: {"real": float(real[key]), "simulated_smooth": float(smooth[key]),
                                           "simulated_jumps": float(jumps[key]), "closer_to": votes[key]}
                                     for key in labels}},
    "chance_accuracy": float(chance),
    "seconds_per_walk": run_seconds / NUM_SEQUENCES,
    "peak_gpu_memory_gb": peak_memory_gb,
}
with open(os.path.join(out_dir, "summary.json"), "w") as f:
    json.dump(summary, f, indent=1)

readme = f"""Results of graph_icl_explorer.ipynb: {MODEL_NAME}, {GRAPH} graph with {num_words} words,
{NUM_SEQUENCES} random walks of {NUM_TOKENS} tokens, block {layer} of {num_layers}, window {window} tokens, seed {SEED}.

accuracy_mean.csv     context = number of words seen; mean_accuracy = probability on the valid next words,
                      averaged over walks; standard_error; belief_fit and two_line_fit = the fitted curves
                      (the two-line fit's onset t_c is in summary.json); mean_revealed_share = share of the
                      current word's neighbours already seen right after it in the walk (lookup reference).
accuracy_binned.csv   the log-binned points that were fitted (and plotted), with the same columns.
fit_parameters.csv    belief-curve parameters p0, q, b, gamma, alpha, N* with 68% bootstrap intervals.
bootstrap_fits.csv    one row per bootstrap refit (walks resampled with replacement).
per_walk_fits.csv     one row per walk: belief-curve fit of that walk alone, transition width (decades between 25%
                      and 75% of the rise), N* from the odd and from the even positions, BIC and held-out error
                      of the belief curve and of the two lines.
single_walk_calibration.csv  the three single-walk diagnostics for the real walks and for the two simulations
                      (smooth walks, sharp jumps), and which simulation the real walks are closer to.
representations.csv   per stored context length: words seen in the window, normalized Dirichlet energy and
                      its random-placement baseline (mean, std of 20 shuffles), PCA R² and its baseline.
pca_coordinates.csv   per stored context length and word: PC1, PC2, and the coordinates lined up with the
                      true drawing (aligned_x, aligned_y; centred, unit total size).
graph_nodes.csv       node number, word, token id, number of neighbours, position in the drawing.
graph_edges.csv       one row per edge.
data.npz              numpy.load("data.npz"): accuracy and revealed_share [walk, position] (position k = after
                      k+1 words), walks [walk, position] (node numbers), context_points, adjacency, layout, words,
                      word_tokens, pca_coordinates and pca_aligned [context point, node, 2] (NaN = not seen),
                      word_counts [context point, node] = occurrences in the window over all walks,
                      word_means [context point, node, hidden size] (float16; only if small enough).
summary.json          settings and main results.
"""
with open(os.path.join(out_dir, "README.txt"), "w") as f:
    f.write(readme)

graph_figure.savefig(os.path.join(out_dir, "graph.png"), dpi=150)
summary_figure.savefig(os.path.join(out_dir, "accuracy_energy_r2.png"), dpi=150)
walk_figure.savefig(os.path.join(out_dir, "single_walks.png"), dpi=150)
pca_figure.savefig(os.path.join(out_dir, "pca_pictures.png"), dpi=150)

zip_path = shutil.make_archive(out_dir, "zip", out_dir)
print(f"Saved {len(os.listdir(out_dir))} files in {out_dir}/ and {zip_path} ({os.path.getsize(zip_path) / 1e6:.1f} MB).")
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)

## 14. Size scan: parameters vs graph size
Set `SCAN_SIZES` in the Settings cell (for example `[3, 4, 5, 6, 8, 10]`) to repeat the accuracy experiment for
several sizes of the same `GRAPH`, with the same model and `NUM_SEQUENCES`. For each size and each entry of
`SCAN_SEEDS`, this cell draws words and walks, runs the model, fits the belief curve and the two lines, bootstraps the
error bars, and computes the revealed share of the same walks (section 8). It then plots every parameter against the
number of words: $p_0$, $q$, $b$, $\gamma$, $\alpha$, the transition point $N^*$ and the two-line onset $t_c$. On the
$N^*$ and $t_c$ panels the revealed share's values are drawn in grey, so the model's timing can be compared with how
fast the walks reveal the graph.

**What the error bars cover.** They come from resampling walks, so they show how much the result depends on the
particular walks, not on the particular words. Each entry of `SCAN_SEEDS` is a separate draw of words (and walks);
use two or three to see how much the choice of words matters. Points from different seeds at the same size are drawn
slightly apart. For `GRAPH = "random"` the seed also changes the graph.

Only accuracy is measured here; the representations are not, which keeps the scan fast. The scan always uses random
words (Park et al.'s words exist only for the 4×4 grid). Bigger graphs need longer walks before the accuracy levels
off; set `SCAN_TOKENS` to give each size its own length. Fits that are not reliable (section 7) are drawn as open
markers and left out of the power-law fits $N^* \approx A \cdot (\text{number of words})^k$, which are straight lines in
log-log axes.

**The revealed share's $t_c$ is not very informative.** The revealed share rises from the first steps, because each
step back to the previous word already reveals a neighbour, so it has no slow phase for the two-line fit to separate.
Its $t_c$ tends to sit near the first context bins and can jump between sizes. On the $t_c$ panel, open and filled
markers, and the choice of points for the power-law fits, follow the reliability of the *belief* fit (as on the $N^*$
panel), for the model and for the revealed share alike; they say nothing about the two-line fit itself. Compare the
model with the revealed share through $N^*$.

The results are saved in `results/<run name>_size_scan/` and downloaded as a zip.

In [ ]:
def run_one_size(graph_size, num_tokens, seed):
    """Run the accuracy experiment for one graph size and one draw of words and walks (set by seed), and fit it.
    Returns (table row, accuracy matrix, revealed-share matrix, walks, fits)."""
    size_rng = np.random.default_rng(seed)
    size_adjacency, _ = build_graph(GRAPH, graph_size, seed)
    size_num_words = len(size_adjacency)
    if size_num_words > len(single_token_words):
        raise ValueError(f"GRAPH_SIZE {graph_size} needs {size_num_words} words; only {len(single_token_words)} available.")
    if max_context is not None and num_tokens + len(bos) > max_context:
        raise ValueError(f"{num_tokens} tokens is longer than the model's context ({max_context}).")
    _, size_tokens, size_neighbour_lists, size_neighbour_tokens = assign_words(size_adjacency, size_rng)

    size_accuracy = np.zeros((NUM_SEQUENCES, num_tokens - 1), dtype=np.float32)
    size_share = np.zeros((NUM_SEQUENCES, num_tokens - 1), dtype=np.float32)
    size_walks = np.zeros((NUM_SEQUENCES, num_tokens), dtype=np.int16)
    for s in tqdm(range(NUM_SEQUENCES), desc=f"{GRAPH} {size_num_words} words, {num_tokens} tokens, seed {seed}"):
        walk = random_walk(num_tokens, size_neighbour_lists, size_rng)
        size_accuracy[s], _ = run_model_on_walk(walk, size_tokens, size_neighbour_tokens, keep_activations=False)
        size_share[s] = revealed_share(walk, size_neighbour_lists)
        size_walks[s] = walk

    x, y = log_binned_mean(size_accuracy)
    belief = fit_belief_curve(x, y, size_rng)
    lines = fit_two_lines(x, y)
    boot = []                                     # bootstrap over walks, for the error bars
    for _ in range(NUM_BOOTSTRAP if NUM_SEQUENCES >= 2 else 0):
        resampled = size_accuracy[size_rng.integers(NUM_SEQUENCES, size=NUM_SEQUENCES)]
        xb, yb = log_binned_mean(resampled)
        refit = fit_belief_curve(xb, yb, size_rng, restarts=8, starts=[belief["x"]])
        refit["t_c"] = fit_two_lines(xb, yb)["t_c"]
        boot.append(refit)
    reliable, progress, problems = fit_reliability(belief, size_accuracy, (x[0], num_tokens))
    _, _, lookup_belief, lookup_lines, lookup_ok, _ = fit_lookup(size_share, np.random.default_rng(seed + 3))

    last = max(20, (num_tokens - 1) // 50)        # positions averaged for the end-of-walk values
    row = {"graph_size": graph_size, "num_words": size_num_words, "num_tokens": num_tokens, "seed": seed,
           "final_accuracy": size_accuracy[:, -last:].mean(), "progress_towards_plateau": progress,
           "fit_reliable": reliable, "fit_problems": "; ".join(problems),
           "two_line_has_plateau_line": lines["num_params"] == 6,
           "N*_lookup": lookup_belief["N*"], "t_c_lookup": lookup_lines["t_c"], "lookup_fit_reliable": lookup_ok,
           "final_revealed_share": size_share[:, -last:].mean()}
    for name in ["p0", "q", "b", "gamma", "alpha", "N*", "t_c"]:
        row[name] = lines["t_c"] if name == "t_c" else belief[name]
        values = np.array([f[name] for f in boot], dtype=float)
        values = values[np.isfinite(values)]
        row[name + "_ci68_low"] = np.percentile(values, 16) if len(values) else np.nan
        row[name + "_ci68_high"] = np.percentile(values, 84) if len(values) else np.nan
    row["N*_model_over_lookup"] = row["N*"] / row["N*_lookup"]
    return row, size_accuracy, size_share, size_walks, (x, y, belief, lines)


if not SCAN_SIZES:
    print("SCAN_SIZES is empty: no size scan. Set it in the Settings cell, e.g. SCAN_SIZES = [3, 4, 5, 6, 8].")
else:
    scan_tokens = SCAN_TOKENS if SCAN_TOKENS is not None else [NUM_TOKENS] * len(SCAN_SIZES)
    if len(scan_tokens) != len(SCAN_SIZES):
        raise ValueError("SCAN_TOKENS needs one value per entry of SCAN_SIZES.")
    if not SCAN_SEEDS or len(set(SCAN_SEEDS)) != len(SCAN_SEEDS) or not all(0 <= s < 1000 for s in SCAN_SEEDS):
        raise ValueError("SCAN_SEEDS must be a list of different whole numbers between 0 and 999, e.g. [0, 1, 2].")
    scan_rows, scan_arrays, scan_fits = [], {}, {}
    for graph_size, num_tokens in zip(SCAN_SIZES, scan_tokens):
        for word_seed in SCAN_SEEDS:
            row, size_accuracy, size_share, size_walks, fits = run_one_size(
                graph_size, int(num_tokens), SEED + 1000 * graph_size + word_seed)
            row["word_seed"] = word_seed
            scan_rows.append(row)
            key = f"{row['num_words']}_words_seed{word_seed}"
            scan_arrays.update({f"accuracy_{key}": size_accuracy, f"revealed_share_{key}": size_share,
                                f"walks_{key}": size_walks})
            scan_fits[(row["num_words"], word_seed)] = fits
            print(f"  {row['num_words']} words, seed {word_seed}: N* = {row['N*']:.0f} (revealed share: "
                  f"{row['N*_lookup']:.0f}), t_c = {row['t_c']:.0f}, q = {row['q']:.2f}, fit "
                  + ("reliable" if row["fit_reliable"] else f"NOT reliable ({row['fit_problems']})"))
    scan_table = pd.DataFrame(scan_rows)
    with pd.option_context("display.float_format", "{:.4g}".format):
        display(scan_table[["graph_size", "num_words", "word_seed", "num_tokens", "p0", "q", "b", "gamma", "alpha",
                            "N*", "t_c", "N*_lookup", "N*_model_over_lookup", "progress_towards_plateau",
                            "fit_reliable"]])

In [ ]:
def draw_power_law(ax, x, values, usable, colour, style, label):
    """Fit values = A * x^k on the usable points (a straight line in log-log), draw it and return (A, k)."""
    usable = usable & np.isfinite(values) & (values > 0)
    if usable.sum() < 3 or len(set(x[usable])) < 2:
        return None
    k, log_a = np.polyfit(np.log10(x[usable]), np.log10(values[usable]), 1)
    xs = np.geomspace(x.min(), x.max(), 50)
    ax.plot(xs, 10 ** log_a * xs ** k, style, color=colour, lw=1, label=f"{label}: {10 ** log_a:.2f}·words^{k:.2f}")
    return 10 ** log_a, k


if SCAN_SIZES:
    num_words_axis = scan_table["num_words"].to_numpy()
    reliable = scan_table["fit_reliable"].to_numpy(dtype=bool)
    lookup_ok = scan_table["lookup_fit_reliable"].to_numpy(dtype=bool)
    seed_index = scan_table["word_seed"].map({s: j for j, s in enumerate(SCAN_SEEDS)}).to_numpy()
    x_plot = num_words_axis * 1.04 ** (seed_index - (len(SCAN_SEEDS) - 1) / 2)   # seeds drawn slightly apart

    # ---- the accuracy curves of all sizes (one colour per size; all seeds of a size in the same colour)
    scan_curve_figure, ax = plt.subplots(figsize=(8, 5))
    sizes = sorted(set(num_words_axis))
    colour_of = dict(zip(sizes, plt.cm.viridis(np.linspace(0, 0.9, len(sizes)))))
    for (n_words, word_seed), (x, y, belief, lines) in scan_fits.items():
        ax.plot(x, y, "o", ms=3, color=colour_of[n_words])
        x_smooth = np.geomspace(x[0], x[-1], 300)
        ax.plot(x_smooth, belief_curve(x_smooth, belief["p0"], belief["q"], belief["b"], belief["gamma"],
                                       belief["alpha"]), color=colour_of[n_words], lw=1.5,
                label=f"{n_words} words" if word_seed == SCAN_SEEDS[0] else None)
    ax.set_xscale("log")
    ax.set_ylim(0, 1)
    ax.set_xlabel("context length (tokens)")
    ax.set_ylabel("rule-following accuracy")
    ax.set_title(f"{MODEL_NAME}, {GRAPH} graphs: accuracy and belief-curve fits", fontsize=10)
    ax.legend(fontsize=8)
    plt.show()

    # ---- every parameter against the number of words
    panels = [("p0", "p0: accuracy before the transition", False), ("q", "q: plateau accuracy", False),
              ("b", "b: prior log-odds", False), ("gamma", "γ: evidence rate", True),
              ("alpha", "α: diminishing returns", False), ("N*", "N*: midpoint of the rise (belief curve)", True),
              ("t_c", "t_c: start of the rise (two-line fit)", True)]
    scan_parameter_figure, axes = plt.subplots(2, 4, figsize=(18, 8))
    power_laws = {}
    for ax, (name, title, log_y) in zip(axes.flat, panels):
        value = scan_table[name].to_numpy(dtype=float)
        low = value - scan_table[name + "_ci68_low"].to_numpy(dtype=float)
        high = scan_table[name + "_ci68_high"].to_numpy(dtype=float) - value
        error = np.nan_to_num(np.vstack([low, high]).clip(min=0))
        for is_reliable, marker_face in [(True, "tab:blue"), (False, "white")]:
            chosen = reliable == is_reliable
            if chosen.any():
                ax.errorbar(x_plot[chosen], value[chosen], yerr=error[:, chosen], fmt="o", capsize=3,
                            color="tab:blue", markerfacecolor=marker_face,
                            label="model, reliable fit" if is_reliable else "model, fit not reliable")
        if name in ("N*", "t_c"):
            result = draw_power_law(ax, num_words_axis, value, reliable, "tab:blue", "--", "model")
            if result:
                power_laws[f"{name}, model"] = result
            lookup_value = scan_table[name + "_lookup"].to_numpy(dtype=float)
            for is_ok, marker_face in [(True, "0.5"), (False, "white")]:
                chosen = lookup_ok == is_ok
                if chosen.any():
                    ax.plot(x_plot[chosen], lookup_value[chosen], "s", color="0.5", markerfacecolor=marker_face,
                            label="revealed share" + ("" if is_ok else ", fit not reliable"))
            result = draw_power_law(ax, num_words_axis, lookup_value, lookup_ok, "0.5", ":", "revealed share")
            if result:
                power_laws[f"{name}, revealed share"] = result
        ax.set_xscale("log")
        if log_y:
            ax.set_yscale("log")
        ax.set_xlabel("number of words")
        ax.set_title(title, fontsize=10)
        ax.legend(fontsize=7)
    ax = axes.flat[7]                                # last panel: how far each run got towards its plateau
    ax.plot(x_plot, scan_table["progress_towards_plateau"], "o", color="0.4")
    ax.axhline(RELIABLE_PROGRESS, color="tab:red", ls=":", lw=1,
               label=f"reliability threshold ({RELIABLE_PROGRESS:.0%})")
    ax.set_xscale("log")
    ax.set_ylim(0, 1.05)
    ax.set_xlabel("number of words")
    ax.set_title("progress towards the plateau at the end of the walk", fontsize=10)
    ax.legend(fontsize=7)
    scan_parameter_figure.suptitle(f"{MODEL_NAME}, {GRAPH} graphs, {NUM_SEQUENCES} walks per run, seeds {SCAN_SEEDS} "
                                   "(error bars: 68% bootstrap intervals over walks)", fontsize=11)
    scan_parameter_figure.tight_layout()
    plt.show()
    for name, (a, k) in power_laws.items():
        print(f"{name}: ≈ {a:.3g} × (number of words)^{k:.2f}  (reliable fits only)")
    ratio = scan_table[scan_table.fit_reliable & scan_table.lookup_fit_reliable].groupby("num_words")["N*_model_over_lookup"]
    if len(ratio):
        print("N*(model) / N*(revealed share), median over seeds:",
              ", ".join(f"{n_words} words: {value:.1f}" for n_words, value in ratio.median().items()))

    # ---- save the scan
    scan_dir = out_dir + "_size_scan"
    os.makedirs(scan_dir, exist_ok=True)
    scan_table.to_csv(os.path.join(scan_dir, "scan_parameters.csv"), index=False)
    curve_rows = []
    for (n_words, word_seed), (x, y, belief, lines) in scan_fits.items():
        fitted = belief_curve(x, belief["p0"], belief["q"], belief["b"], belief["gamma"], belief["alpha"])
        for xi, yi, bi, li in zip(x, y, fitted, two_lines_curve(x, lines)):
            curve_rows.append({"num_words": n_words, "word_seed": word_seed, "context_bin_centre": xi,
                               "mean_accuracy": yi, "belief_fit": bi, "two_line_fit": li})
    pd.DataFrame(curve_rows).to_csv(os.path.join(scan_dir, "scan_accuracy_binned.csv"), index=False)
    np.savez_compressed(os.path.join(scan_dir, "scan_data.npz"), **scan_arrays)
    with open(os.path.join(scan_dir, "README.txt"), "w") as f:
        f.write(f"Size scan of graph_icl_explorer.ipynb: {MODEL_NAME}, {GRAPH} graphs, sizes {SCAN_SIZES}, "
                f"word seeds {SCAN_SEEDS}, {NUM_SEQUENCES} walks per run, SEED {SEED}.\n"
                "scan_parameters.csv      one row per size and word seed: belief-curve parameters p0, q, b, gamma,\n"
                "                         alpha, N*, the two-line onset t_c, each with 68% bootstrap intervals over walks;\n"
                "                         whether the fit is reliable (and why not); the revealed share's N* and t_c\n"
                "                         (lookup reference) and N*_model_over_lookup. seed = SEED + 1000 x GRAPH_SIZE\n"
                "                         + word seed, the random seed of that run's words and walks.\n"
                "scan_accuracy_binned.csv the log-binned mean accuracy of each run with both fitted curves.\n"
                "scan_data.npz            for each run (key suffix <N>_words_seed<s>): accuracy_..., revealed_share_...\n"
                "                         [walk, position] (position k = after k + 1 words) and walks_... [walk, position].\n")
    scan_curve_figure.savefig(os.path.join(scan_dir, "scan_accuracy_curves.png"), dpi=150)
    scan_parameter_figure.savefig(os.path.join(scan_dir, "scan_parameters.png"), dpi=150)
    scan_zip = shutil.make_archive(scan_dir, "zip", scan_dir)
    print(f"Saved {scan_zip} ({os.path.getsize(scan_zip) / 1e6:.1f} MB)")
    if IN_COLAB:
        from google.colab import files
        files.download(scan_zip)

## How to read the results
- **Accuracy and the two fits.** The accuracy starts near chance and rises as the model picks up the graph. Park et al.
  describe the rise as two straight lines on a log axis; the belief curve is an S-shape on the same axis. Compare them
  by held-out error first. Remember that $t_c$ marks where the rise starts and $N^*$ its midpoint.
- **Reliability.** If the accuracy is still rising at the end of the walk, $q$ ends up at its upper limit 1 and $b$,
  $\gamma$, $\alpha$ trade off against each other, so individual parameters mean little. Run longer walks: the warning
  in section 7 suggests a length. Bigger graphs need much longer walks.
- **Lookup reference.** The revealed share shows when the walk has shown enough transitions for a memoriser to use.
  A large ratio $N^*(\text{model}) / N^*(\text{revealed share})$ means the model needs to see transitions several
  times; similar growth of the two $N^*$ with graph size means the timing of the transition follows how fast the walk
  reveals the graph. Compare timing and growth, not heights (section 8), and use $N^*$ rather than $t_c$ for the
  revealed share (section 14).
- **Single walks.** Section 9 says whether the S-shape belongs to the individual walks or comes from averaging. Look at
  the table, not only at the verdict: it shows how far apart the two simulations are on each diagnostic.
- **Energy and pictures.** Energy below its baseline means that neighbouring words have more similar mean activations
  than randomly placed words would. On large graphs, read it together with the caveats of section 10.
- **Random graph.** It has no spatial layout; its pictures are compared with its spectral drawing.
- **Words.** All walks in a run share one set of words, as in Park et al. Run again with other `SEED` values (or several
  `SCAN_SEEDS` in the scan) to see how much the results depend on the words.

## Loading the saved data later
```python
import numpy as np, pandas as pd
folder = "results/<run name>"                     # the unzipped folder
accuracy = pd.read_csv(f"{folder}/accuracy_mean.csv")
reps = pd.read_csv(f"{folder}/representations.csv")
data = np.load(f"{folder}/data.npz")
data["accuracy"].shape                            # (walks, positions)
data["revealed_share"].shape                      # (walks, positions)
data["word_means"].shape                          # (context points, words, hidden size), if saved
```

## Technical details and reproducibility

**Software.** The notebook is written for Google Colab's GPU runtime (Python 3). In Colab, the setup cell (section 1)
upgrades `transformers` and `accelerate` to their newest versions; Transformers must be at least 4.56, which the
notebook checks. PyTorch, NumPy, SciPy, pandas, Matplotlib, NetworkX and tqdm are Colab's preinstalled versions, so
the versions used depend on the date of the run. The model is loaded with `AutoModelForCausalLM` and runs through
the standard Hugging Face forward pass; the activations are read with a forward hook on the chosen block.

**Model revisions and environment.** The weights are downloaded at their latest revision (`main`). `summary.json`
records the model, the precision and every setting, but not the model revision, the GPU or the library versions. To
keep them with your results, run this in a new cell after the model is loaded:
```python
import torch, transformers
print(model.config._commit_hash, transformers.__version__, torch.__version__,
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
```
The runs reported in the write-up were made on Google Colab in November 2026, on an NVIDIA L4 (bfloat16) for
Llama-3.1-8B and other 7–8B models and on a T4 for 1–1.5B models, with Colab's runtime of that month (PyTorch 2.14.1
with CUDA 13.0), the newest Transformers release then available, and the latest revision of each model on Hugging
Face.

**Precision.** With `DTYPE = "auto"`, models run in bfloat16 on GPUs of compute capability 8 or higher (L4, A100), in
float16 on older GPUs (T4) and in float32 on the CPU. Gemma runs in float32 instead of float16. The notebook stops if
a model produces infinite or NaN values. Runs on different GPUs or library versions agree up to small floating-point
differences.

**Random numbers.** All randomness follows from `SEED`: the words and walks use `SEED`, the belief-curve fits and
their bootstrap `SEED + 1`, the baselines of the representations `SEED + 2`, the revealed-share fit `SEED + 3` and the single-walk
simulations `SEED + 4`. In the size scan, each size and word seed uses `SEED + 1000 × GRAPH_SIZE + word seed`. For
`GRAPH = "random"`, the graph is a random 4-regular graph drawn from the same seed (redrawn with seed + 1000 in the
rare case that it is not connected).

**Graphs.** The square, triangular (rhombus) and hexagonal graphs are built exactly as in the companion head-ablation
notebook. The ring and the random graph have n² words.

**Tests of the code.** The analysis was run end to end on CPU for all five graphs, including the size scan, with a
simulated model in place of the language model (NumPy 2.5.3, SciPy 1.18.1, pandas 3.0.5, Matplotlib 3.11.2,
NetworkX 3.7). These tests replace the cells that load and run the model and compute the representations with
simulated data, so they cover the fits, the bootstrap, the held-out comparison, the lookup reference, the single-walk
check, the figures and the saved files.

### References

[Park et al., *ICLR: In-Context Learning of Representations*](https://arxiv.org/abs/2501.00070) · [Kowalyshyn et al., *Belief or Circuitry? Causal Evidence for In-Context Graph Learning*](https://arxiv.org/abs/2605.08405)

Software: [PyTorch](https://pytorch.org) · [Hugging Face Transformers](https://github.com/huggingface/transformers) · [NetworkX](https://networkx.org) · [NumPy](https://numpy.org) · [SciPy](https://scipy.org) · [pandas](https://pandas.pydata.org) · [Matplotlib](https://matplotlib.org). Each model is subject to the licence on its Hugging Face page; Llama and Gemma models are gated.